# Supplementary Code — Long-term meteorological drought in the Cholistan Desert, Pakistan (1981–2025)

Reproducible analysis of CHIRPS rainfall for the Cholistan Desert: rainfall regime, Standardized Precipitation Index (SPI), trends and change points, dry-pentad and wet-pentad behaviour, pixel-wise spatial trends with Local Moran's I clusters, and robustness to dataset version (CHIRPS v2 vs v3) and to independent precipitation products.

**Inputs** (relative to the project folder)
| File | Content |
|---|---|
| `02_Data/Regional/CHIRPS_pentad_area_mean_1981_2025_RAW_CACHE.csv` | area-weighted CHIRPS v2 pentad rainfall over the Cholistan boundary (Section 4 rebuilds it from Earth Engine) |
| `03_GIS/Cholistan_Boundary/Cholistan_Boundary.shp` | study-area boundary (WGS 84) |
| `02_Data/Spatial_Cache/` (Section 1) | native 0.05° CHIRPS v2/v3 pentad cubes and monthly series of PERSIANN-CDR, ERA5-Land and IMERG V07 (rebuilt from Earth Engine when `ALLOW_EARTH_ENGINE = True`) |
| `03_GIS/OSM_Geofabrik/pakistan-latest-free.shp.zip` (optional) | OpenStreetMap context for the study-area map |

**Requirements:** Python ≥ 3.10 with `numpy pandas scipy scikit-learn matplotlib pillow`; optional `earthengine-api` (rebuilding caches), `geopandas` (OSM map), `rasterio` (GeoTIFF export). All statistics are implemented in this notebook (Section 2) and checked against known answers (Section 3).

**Run:** set `PROJECT_ROOT` in Section 1 (or the environment variable `CHOLISTAN_PROJECT_ROOT`) and run all cells in order. Runtime is ≈ 5–10 min, mostly permutation and bootstrap inference.

**Outputs** (`05_Outputs/`): `Main/` and `Supplementary/` figures (600-dpi LZW-compressed RGB TIFF at final printed size: 174 mm double-column width, ≤ 230 mm height) and tables (CSV); `GIS_rasters/` (optional GeoTIFF); `Captions.md`, `Key_results.md`, `Output_manifest.csv`, `Figure_quality_check.csv` and `Provenance.json`. Captions and key results are generated from the computed values, so they always match the figures and tables.

**Reproducibility:** fixed random seed (2026); 2,000 block-bootstrap and 9,999 permutation replicates; no step overwrites the input files.

In [ ]:
# =============================================================================
# 1. Configuration — the only cell that needs editing
# =============================================================================

import io
import json
import os
import sys
import time
import platform
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy import stats, special

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)


# =============================================================================
# Project paths
# =============================================================================

PROJECT_ROOT = Path(
    os.environ.get(
        "CHOLISTAN_PROJECT_ROOT",
        (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    )
).expanduser()

# Regional rainfall data
CSV_IN = PROJECT_ROOT / "data" / "regional"

PENTAD_CACHE_CSV = (
    CSV_IN / "CHIRPS_pentad_area_mean_1981_2025_RAW_CACHE.csv"
)

# Study-area boundary
BOUNDARY_SHP = (
    PROJECT_ROOT
    / "data" / "gis"
    / "Cholistan_Boundary"
    / "Cholistan_Boundary.shp"
)

# Optional OpenStreetMap data
OSM_ZIP = (
    PROJECT_ROOT
    / "data" / "gis"
    / "OSM_Geofabrik"
    / "pakistan-latest-free.shp.zip"
)

# Native-grid rainfall and precipitation-product caches
SPATIAL_CACHE = PROJECT_ROOT / "data" / "spatial_cache"

# Publication-ready outputs
OUT = PROJECT_ROOT / "outputs"

# Cached Earth Engine extracts
# Input datasets are kept separate from generated publication outputs.

CACHE_CANDIDATES = (
    [Path(os.environ["CHOLISTAN_SPATIAL_CACHE"]).expanduser()]
    if os.environ.get("CHOLISTAN_SPATIAL_CACHE")
    else []
)

CACHE_CANDIDATES += [SPATIAL_CACHE]


# =============================================================================
# Earth Engine configuration
# Only required when rebuilding missing caches from source
# =============================================================================

ALLOW_EARTH_ENGINE = False

EE_PROJECT_ID = os.environ.get("EE_PROJECT_ID", "")  # your Google Cloud project with Earth Engine enabled

EE_BOUNDARY_ASSET = (
    os.environ.get("EE_BOUNDARY_ASSET", "")  # optional: Earth Engine asset of the boundary
)

# Primary precipitation dataset
CHIRPS_V2_PENTAD = "UCSB-CHG/CHIRPS/PENTAD"

# Version sensitivity dataset
CHIRPS_V3_PENTAD = "UCSB-CHC/CHIRPS/V3/PENTAD"

# Independent precipitation products for agreement analysis
# These products are not treated as ground truth.

PRODUCTS = {
    "PERSIANN-CDR": dict(
        id="NOAA/PERSIANN-CDR",
        band="precipitation",
        years=(1983, 2025),
        scale=27830,
        kind="daily_mm"
    ),

    "ERA5-Land": dict(
        id="ECMWF/ERA5_LAND/MONTHLY_AGGR",
        band="total_precipitation_sum",
        years=(1981, 2025),
        scale=11132,
        kind="monthly_m"
    ),

    "IMERG-V07": dict(
        id="NASA/GPM_L3/IMERG_MONTHLY_V07",
        band="precipitation",
        years=(2001, 2024),
        scale=11132,
        kind="monthly_mm_per_hr"
    ),
}


# =============================================================================
# Study design and statistical parameters
# =============================================================================

START_YEAR, END_YEAR = 1981, 2025

YEARS = np.arange(
    START_YEAR,
    END_YEAR + 1
)

# WMO climatological standard normal
NORMAL = (1991, 2020)

# None = full 1981–2025 record
SPI_PRIMARY_CALIBRATION = None

SPI_SENSITIVITY_CALIBRATION = NORMAL

SPI_SCALES = (1, 3, 6, 12)

SPI_SCALES_MAIN = (3, 6, 12)

# JJAS primary, JAS supplementary
SEASONS = {
    "JJAS": (6, 7, 8, 9),
    "JAS": (7, 8, 9)
}

# Candidate break tested in Section 6
CHANGE_YEAR = 2009

SUBPERIODS = {
    "1981–2025": (1981, 2025),
    "1981–2008": (1981, 2008),
    "1981–2009": (1981, 2009),
    "2010–2025": (2010, 2025),
    "1991–2025": (1991, 2025),
    "1981–2020": (1981, 2020)
}

# Wet-pentad threshold
WET_PENTAD_MM = 5.0

# Calendar pentads with a 1991–2020 mean below this threshold
NEAR_RAINLESS_MM = 1.0

# Alternative dry-pentad definitions
DRY_DEFINITIONS = {

    "P20 (all pentads)": dict(
        kind="percentile",
        q=0.20
    ),

    "P25 (all pentads)": dict(
        kind="percentile",
        q=0.25
    ),

    "P10 (all pentads)": dict(
        kind="percentile",
        q=0.10
    ),

    "P05 (all pentads)": dict(
        kind="percentile",
        q=0.05
    ),

    "P20 (normal ≥1 mm)": dict(
        kind="percentile",
        q=0.20,
        min_clim_mm=1.0
    ),

    "P20 (normal ≥2 mm)": dict(
        kind="percentile",
        q=0.20,
        min_clim_mm=2.0
    ),

    "Absolute <1 mm": dict(
        kind="absolute",
        thr_mm=1.0
    ),

    "Absolute <2 mm": dict(
        kind="absolute",
        thr_mm=2.0
    ),
}

# Minimum fraction of a native grid cell inside the study area
ANALYSIS_MIN_FRACTION = 0.5

# Spatial autocorrelation permutations
LISA_PERMUTATIONS = 9999

# Bootstrap iterations, permutation iterations and random seed
N_BOOT, N_PERM, SEED = 2000, 9999, 2026


# =============================================================================
# Output directory configuration
# =============================================================================

DIRS = {
    k: OUT / v
    for k, v in dict(
        main_fig="Main/figures",
        main_tab="Main/tables",
        supp_fig="Supplementary/figures",
        supp_tab="Supplementary/tables",
        gis="GIS_rasters"
    ).items()
}

# Input caches remain outside the publication output directory
DIRS["cache"] = SPATIAL_CACHE

# Confirm that the project root exists before creating directories
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project root does not exist:\n{PROJECT_ROOT}"
    )

# Create required directories if they do not already exist
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Input data verification
# =============================================================================

REQUIRED_INPUTS = {

    "Regional pentad rainfall":
        PENTAD_CACHE_CSV,

    "SPI validation data":
        CSV_IN / "spi_gamma_primary.csv",

    "Cholistan boundary (.shp)":
        BOUNDARY_SHP,

    "Cholistan boundary (.shx)":
        BOUNDARY_SHP.with_suffix(".shx"),

    "Cholistan boundary (.dbf)":
        BOUNDARY_SHP.with_suffix(".dbf"),

    "Cholistan boundary (.prj)":
        BOUNDARY_SHP.with_suffix(".prj"),
}


# -----------------------------------------------------------------------------
# Identify the active spatial cache
# Match the cache-selection logic used in Section 10.
# -----------------------------------------------------------------------------

ACTIVE_CACHE = next(
    (
        folder
        for folder in CACHE_CANDIDATES

        if (
            folder / "chirps_native_grid.json"
        ).is_file()

        and (
            (
                folder
                / f"chirps_v2_cube_{START_YEAR}_{END_YEAR}.npz"
            ).is_file()

            or all(
                (
                    folder / f"chirps_v2_{int(year)}.npy"
                ).is_file()
                for year in YEARS
            )
        )
    ),
    SPATIAL_CACHE
)


# -----------------------------------------------------------------------------
# Verify consolidated rainfall datasets for the offline workflow
# -----------------------------------------------------------------------------

if not ALLOW_EARTH_ENGINE:

    REQUIRED_INPUTS.update({

        "CHIRPS native grid metadata":
            ACTIVE_CACHE
            / "chirps_native_grid.json",

        "CHIRPS v2 pentad cube":
            ACTIVE_CACHE
            / f"chirps_v2_cube_{START_YEAR}_{END_YEAR}.npz",

        "CHIRPS v3 pentad cube":
            ACTIVE_CACHE
            / f"chirps_v3_cube_{START_YEAR}_{END_YEAR}.npz",
    })

    # Independent precipitation-product caches
    for name in PRODUCTS:

        filename = f"interproduct_{name}_monthly.csv"

        REQUIRED_INPUTS[name] = next(
            (
                folder / filename

                for folder in [
                    ACTIVE_CACHE,
                    *CACHE_CANDIDATES
                ]

                if (folder / filename).is_file()
            ),
            ACTIVE_CACHE / filename
        )


# -----------------------------------------------------------------------------
# Display verification results
# -----------------------------------------------------------------------------

print("=" * 78)
print("INPUT DATA VERIFICATION")
print("=" * 78)

missing_inputs = []

for label, path in REQUIRED_INPUTS.items():

    if path.is_file():

        print(f"[OK]       {label}")

    else:

        print(f"[MISSING]  {label}")

        missing_inputs.append(path)

print("-" * 78)

# Stop execution before starting the analysis if inputs are missing
if missing_inputs:

    raise FileNotFoundError(
        "Required input files are missing:\n"
        + "\n".join(str(p) for p in missing_inputs)
    )

print("All required input files are available.")


# -----------------------------------------------------------------------------
# Optional mapping data
# -----------------------------------------------------------------------------

if OSM_ZIP.is_file():

    print("[OK]       Optional OpenStreetMap archive available.")

else:

    print("[OPTIONAL] OpenStreetMap data not available.")
    print("           Boundary-and-grid study-area map will be generated.")


# -----------------------------------------------------------------------------
# Display project configuration
# -----------------------------------------------------------------------------

print("-" * 78)
print("PROJECT CONFIGURATION")
print("-" * 78)

print("Project root       :", PROJECT_ROOT)
print("Regional data      :", CSV_IN)
print("Spatial cache      :", ACTIVE_CACHE)
print("GIS boundary       :", BOUNDARY_SHP)
print("Publication outputs:", OUT)

print("-" * 78)

print("Study period       :", f"{START_YEAR}–{END_YEAR}")
print("Reference period   :", f"{NORMAL[0]}–{NORMAL[1]}")
print("Number of years    :", len(YEARS))
print("SPI scales         :", SPI_SCALES)
print("Primary monsoon    :", "JJAS")
print("Earth Engine       :", "Enabled" if ALLOW_EARTH_ENGINE else "Disabled")

print("=" * 78)


# =============================================================================
# Journal figure specifications
# =============================================================================
# Original artwork configuration retained:
#   Single-column width:    85 mm
#   1.5-column width:      120 mm
#   Double-column width:   174 mm
#   Maximum figure height: 230 mm
#   Resolution:            600 dpi
#
# Figures are saved at their stated physical dimensions without tight cropping.
# =============================================================================

MM = 1 / 25.4

FIG_WIDTH_MM = {
    "single": 85,
    "onehalf": 120,
    "double": 174
}

FIG_MAX_HEIGHT_MM = 230

FIG_DPI = 600


# -----------------------------------------------------------------------------
# Publication colour palette
# -----------------------------------------------------------------------------

PAL = dict(
    ink="#231F23",
    muted="#5E565E",
    purple="#91278F",
    purple_dark="#5F175E",
    purple_mid="#B468B2",
    purple_light="#E6D0E5",
    grey="#A9A3A9",
    grey_light="#DCD8DC"
)

PRODUCT_COLORS = {
    "CHIRPS v2": PAL["purple_dark"],
    "CHIRPS v3": PAL["purple_mid"],
    "PERSIANN-CDR": "#2B8C8C",
    "ERA5-Land": "#C9822B",
    "IMERG-V07": "#4F6DB8"
}


# -----------------------------------------------------------------------------
# Matplotlib publication styling
# -----------------------------------------------------------------------------

mpl.rcParams.update({

    # Typography
    "font.family": "sans-serif",

    "font.sans-serif": [
        "Arial",
        "Helvetica",
        "Liberation Sans",
        "DejaVu Sans"
    ],

    "font.size": 8,
    "axes.titlesize": 8.5,
    "axes.labelsize": 8,

    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,

    # Axes and tick styling
    "axes.linewidth": 0.6,

    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,

    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,

    "lines.linewidth": 1.0,

    "axes.spines.top": False,
    "axes.spines.right": False,

    "axes.edgecolor": PAL["muted"],

    "xtick.color": PAL["muted"],
    "ytick.color": PAL["muted"],

    "axes.labelcolor": PAL["ink"],
    "text.color": PAL["ink"],

    "axes.titlelocation": "left",

    # Figure and legend styling
    "legend.frameon": False,
    "figure.dpi": 100,

    "pdf.fonttype": 42,

    "figure.constrained_layout.h_pad": 0.03,
    "figure.constrained_layout.w_pad": 0.03,
})


# =============================================================================
# Output registries
# Used for captions, output manifests and quality control
# =============================================================================

FIGURES = []
TABLES = []


# =============================================================================
# Figure and table helper functions
# =============================================================================

def new_figure(width="double", height_mm=100, **kw):
    """
    Create a publication figure at its final printed physical dimensions.

    Parameters
    ----------
    width : str
        Figure width class: single, onehalf or double.

    height_mm : float
        Figure height in millimetres.

    **kw : dict
        Additional Matplotlib figure arguments.
    """

    if height_mm > FIG_MAX_HEIGHT_MM:

        raise ValueError(
            f"Height {height_mm} mm exceeds "
            f"{FIG_MAX_HEIGHT_MM} mm"
        )

    return plt.figure(
        figsize=(
            FIG_WIDTH_MM[width] * MM,
            height_mm * MM
        ),
        layout="constrained",
        **kw
    )


def panel_label(ax, letter, x=-0.02, y=1.02):
    """
    Add a publication-style panel label to a Matplotlib axis.
    """

    ax.text(
        x,
        y,
        letter,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=9.5,
        fontweight="bold"
    )


def save_figure(fig, name, role, caption, width="double"):
    """
    Save a publication-ready figure as a 600-dpi,
    LZW-compressed, flattened RGB TIFF.

    The original figure dimensions are preserved,
    and the figure is registered for documentation
    and quality-control purposes.
    """

    from PIL import Image

    # Select the appropriate output directory
    folder = DIRS[
        "main_fig"
        if role == "Main"
        else "supp_fig"
    ]

    path = folder / f"{name}.tiff"

    # Render figure to an in-memory PNG
    buf = io.BytesIO()

    fig.savefig(
        buf,
        format="png",
        dpi=FIG_DPI,
        facecolor="white"
    )

    buf.seek(0)

    # Convert to compressed RGB TIFF
    with Image.open(buf) as im:

        im.convert("RGB").save(
            path,
            format="TIFF",
            dpi=(FIG_DPI, FIG_DPI),
            compression="tiff_lzw"
        )

    buf.close()

    # Register the generated figure
    FIGURES.append(
        dict(
            name=name,
            role=role,
            width_class=width,
            caption=caption,
            file=str(path.relative_to(OUT))
        )
    )

    plt.close(fig)

    return path


def save_table(df, name, role, caption):
    """
    Save a main or supplementary table as CSV
    and register it for documentation and manifests.
    """

    # Select the appropriate output directory
    folder = DIRS[
        "main_tab"
        if role == "Main"
        else "supp_tab"
    ]

    path = folder / f"{name}.csv"

    # Export the table
    df.to_csv(
        path,
        index=False
    )

    # Register the generated table
    TABLES.append(
        dict(
            name=name,
            role=role,
            caption=caption,
            file=str(path.relative_to(OUT)),
            rows=len(df)
        )
    )

    return path


# =============================================================================
# Runtime and typography information
# =============================================================================

from matplotlib import font_manager

available_fonts = {
    f.name
    for f in font_manager.fontManager.ttflist
}

selected_font = next(
    (
        font
        for font in mpl.rcParams["font.sans-serif"]
        if font in available_fonts
    ),
    "DejaVu Sans"
)

font_path = font_manager.findfont(selected_font)

print()
print("=" * 78)
print("RUNTIME ENVIRONMENT")
print("=" * 78)

print("Python             :", platform.python_version())
print("Operating system   :", platform.system())
print("NumPy              :", np.__version__)
print("Pandas             :", pd.__version__)
print("Matplotlib         :", mpl.__version__)

print("-" * 78)

print("Publication font   :", selected_font)
print("Font file          :", Path(font_path).name)
print("Figure resolution  :", f"{FIG_DPI} dpi")
print("Figure format      :", "RGB TIFF (LZW compression)")
print("Figure widths      :", FIG_WIDTH_MM)

print("-" * 78)

if selected_font == "DejaVu Sans":

    print(
        "Typography note: Arial or Liberation Sans may be "
        "installed if required by the journal."
    )

print("Project            :", PROJECT_ROOT)
print("Outputs            :", OUT)
print("Spatial cache      :", ACTIVE_CACHE)
print("EE downloads       :", ALLOW_EARTH_ENGINE)

print("=" * 78)
print("CONFIGURATION COMPLETED SUCCESSFULLY")
print("Proceed to Section 2.")
print("=" * 78)

In [ ]:
# =============================================================================
# 2. Statistical methods (shared by the regional and spatial analyses)
# -----------------------------------------------------------------------------
# Every function accepts either a 1-D series (n,) or a matrix (n, P) where the
# P columns are independent series (e.g. pixels).  Only NumPy/SciPy are used so
# the same code runs on the regional series and on thousands of CHIRPS pixels.
# =============================================================================
import numpy as np
import pandas as pd
from scipy import stats, special


def _as2d(x):
    x = np.asarray(x, dtype=float)
    return (x[:, None], True) if x.ndim == 1 else (x, False)


def bh_adjust(p):
    """Benjamini–Hochberg (1995) adjusted p-values (q-values); NaNs are ignored."""
    p = np.asarray(p, dtype=float)
    out = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if ok.any():
        pv = p[ok]
        m = pv.size
        order = np.argsort(pv)
        ranked = pv[order] * m / np.arange(1, m + 1)
        ranked = np.minimum.accumulate(ranked[::-1])[::-1]
        adj = np.empty(m)
        adj[order] = np.minimum(ranked, 1.0)
        out[ok] = adj
    return out


def sen_slope(x, t=None):
    """Theil–Sen slope and intercept for each column (complete columns only)."""
    X, one = _as2d(x)
    n = X.shape[0]
    t = np.arange(n, dtype=float) if t is None else np.asarray(t, dtype=float)
    i, j = np.triu_indices(n, 1)
    slopes = (X[j] - X[i]) / (t[j] - t[i])[:, None]
    b = np.median(slopes, axis=0)
    a = np.median(X - b * t[:, None], axis=0)
    return (b[0], a[0]) if one else (b, a)


def _mk_S_and_var(X):
    n, P = X.shape
    i, j = np.triu_indices(n, 1)
    S = np.sign(X[j] - X[i]).sum(axis=0)
    var0 = np.full(P, n * (n - 1) * (2 * n + 5) / 18.0)
    Xs = np.sort(X, axis=0)
    same = np.vstack([np.zeros((1, P), bool), Xs[1:] == Xs[:-1]])
    if same.any():  # tie correction only where ties exist
        for c in np.where(same.any(axis=0))[0]:
            _, cnt = np.unique(X[:, c], return_counts=True)
            var0[c] -= np.sum(cnt * (cnt - 1) * (2 * cnt + 5)) / 18.0
    return S, var0


def mk_hamed_rao(x, t=None, acf_alpha=0.05):
    """Hamed & Rao (1998) modified Mann–Kendall test + Sen slope (vectorised).

    Same definition as the submitted notebook: ranks of Sen-detrended data,
    all lags 1..n-1, only lags significant at `acf_alpha` enter the
    variance-correction factor n/n*.  Returns a dict of arrays (or scalars).
    Also returns diagnostics of serial dependence: lag-1 autocorrelation of the
    detrended series, the variance inflation factor and effective sample size.
    """
    X, one = _as2d(x)
    n, P = X.shape
    t = np.arange(n, dtype=float) if t is None else np.asarray(t, dtype=float)
    S, var0 = _mk_S_and_var(X)
    b, a = sen_slope(X, t)
    resid = X - b * t[:, None]
    R = stats.rankdata(resid, axis=0)
    Rc = R - R.mean(axis=0)
    den = (Rc ** 2).sum(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        acf = np.array([(Rc[:-k] * Rc[k:]).sum(axis=0) / den for k in range(1, n)])
    acf = np.where(np.isfinite(acf), acf, 0.0)
    thr = stats.norm.ppf(1 - acf_alpha / 2) / np.sqrt(n)
    acf_sig = np.where(np.abs(acf) > thr, acf, 0.0)
    k = np.arange(1, n, dtype=float)
    w = (n - k) * (n - k - 1) * (n - k - 2)
    essf = 1.0 + 2.0 / (n * (n - 1) * (n - 2)) * (w[:, None] * acf_sig).sum(axis=0)
    var_c = var0 * essf
    with np.errstate(invalid="ignore", divide="ignore"):
        z_raw = np.where(S == 0, 0.0, (S - np.sign(S)) / np.sqrt(var0))
        z = np.where(S == 0, 0.0, (S - np.sign(S)) / np.sqrt(var_c))
    z = np.where(var_c > 0, z, np.nan)
    z_raw = np.where(var0 > 0, z_raw, np.nan)
    # lag-1 Pearson autocorrelation of the Sen-detrended series (diagnostic)
    rc = resid - resid.mean(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        r1 = (rc[:-1] * rc[1:]).sum(axis=0) / (rc ** 2).sum(axis=0)
    out = dict(
        n=np.full(P, n), S=S, sen_slope=b, sen_intercept=a,
        tau_a=S / (0.5 * n * (n - 1)),
        z_raw=z_raw, p_raw=2 * stats.norm.sf(np.abs(z_raw)),
        z_hr=z, p_hr=2 * stats.norm.sf(np.abs(z)),
        variance_factor_n_over_nstar=essf,
        n_effective=np.where(essf > 0, n / essf, np.nan),
        rank_acf_lag1=acf[0], pearson_acf_lag1_detrended=r1,
        lag1_significant=np.abs(acf[0]) > thr,
    )
    return {k: v[0] for k, v in out.items()} if one else out


def mk_tfpw(x):
    """Trend-free pre-whitening Mann–Kendall (Yue et al. 2002), 1-D sensitivity test."""
    x = np.asarray(x, float)
    n = len(x)
    t = np.arange(n, dtype=float)
    b, _ = sen_slope(x, t)
    r = x - b * t
    rc = r - r.mean()
    r1 = float((rc[:-1] * rc[1:]).sum() / (rc ** 2).sum()) if (rc ** 2).sum() > 0 else 0.0
    y = (r[1:] - r1 * r[:-1]) + b * t[1:]
    S, var0 = _mk_S_and_var(y[:, None])
    z = 0.0 if S[0] == 0 else (S[0] - np.sign(S[0])) / np.sqrt(var0[0])
    return dict(r1=r1, z_tfpw=z, p_tfpw=2 * stats.norm.sf(abs(z)))


def _mbb_indices(n, B, L, rng):
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=(B, nb))
    idx = (starts[:, :, None] + np.arange(L)[None, None, :]).reshape(B, -1)[:, :n]
    return idx


def mbb_sen_ci(x, B=2000, L=None, level=0.95, seed=2026):
    """Moving-block bootstrap CI for the Sen slope (preserves serial dependence)."""
    x = np.asarray(x, float)
    n = len(x)
    t = np.arange(n, dtype=float)
    L = L or max(2, int(round(n ** (1 / 3))))
    b, a = sen_slope(x, t)
    resid = x - (a + b * t)
    rng = np.random.default_rng(seed)
    idx = _mbb_indices(n, B, L, rng)
    Xb = (a + b * t)[:, None] + resid[idx].T
    bb, _ = sen_slope(Xb, t)
    lo, hi = np.quantile(bb, [(1 - level) / 2, 1 - (1 - level) / 2])
    return dict(sen_slope=b, ci_low=lo, ci_high=hi, block_length=L, B=B)


def trend_row(x, years, label, units, B=2000, seed=2026):
    """One complete trend record: Hamed–Rao MK, TFPW, unadjusted MK, Sen slope with a
    moving-block bootstrap CI, lag-1 autocorrelation and effective sample size.
    Non-finite years are dropped; Sen slopes use the actual years (gaps allowed)."""
    x = np.asarray(x, float)
    ok = np.isfinite(x)
    x, yrs = x[ok], np.asarray(years)[ok]
    t = (yrs - yrs[0]).astype(float)
    hr = mk_hamed_rao(x, t=t)
    tf = mk_tfpw(x)
    ci = mbb_sen_ci(x, B=B, seed=seed)
    return dict(series=label, units=units, first_year=int(yrs[0]), last_year=int(yrs[-1]),
                n=int(len(x)), sen_slope_per_year=hr["sen_slope"],
                sen_ci95_low=ci["ci_low"], sen_ci95_high=ci["ci_high"],
                kendall_tau=hr["tau_a"], z_hr=hr["z_hr"], p_hr=hr["p_hr"], p_mk_unadjusted=hr["p_raw"],
                p_tfpw=tf["p_tfpw"], lag1_acf_detrended=hr["pearson_acf_lag1_detrended"],
                lag1_significant=bool(hr["lag1_significant"]),
                variance_factor=hr["variance_factor_n_over_nstar"], n_effective=hr["n_effective"])


def trend_family(rows, family):
    df = pd.DataFrame(rows)
    df.insert(0, "family", family)
    df["q_bh_within_family"] = bh_adjust(df["p_hr"])
    df["significant_q05"] = df["q_bh_within_family"] < 0.05
    return df


# ---------------------------------------------------------------- SPI ------
def lmom_gamma_fit(X):
    """Two-parameter gamma by L-moments (Hosking 1990 rational approximation,
    identical to lmoments3.distr.gam.lmom_fit). Columns may contain NaN."""
    X, one = _as2d(X)
    Xs = np.sort(X, axis=0)                      # NaN sorted to the end
    valid = np.isfinite(Xs)
    nk = valid.sum(axis=0).astype(float)
    i = np.arange(X.shape[0], dtype=float)[:, None]
    with np.errstate(invalid="ignore", divide="ignore"):
        b0 = np.nansum(Xs, axis=0) / nk
        b1 = np.nansum(np.where(valid, Xs * i / (nk - 1), 0.0), axis=0) / nk
        l1, l2 = b0, 2 * b1 - b0
        tcv = l2 / l1
        z1 = np.pi * tcv ** 2
        a_small = (1 - 0.3080 * z1) / (z1 - 0.05812 * z1 ** 2 + 0.01765 * z1 ** 3)
        z2 = 1 - tcv
        a_large = (0.7213 * z2 - 0.5947 * z2 ** 2) / (1 - 2.1817 * z2 + 1.2113 * z2 ** 2)
        alpha = np.where(tcv < 0.5, a_small, a_large)
        beta = l1 / alpha
    bad = (nk < 4) | ~np.isfinite(alpha) | (alpha <= 0) | ~(beta > 0)
    alpha[bad], beta[bad] = np.nan, np.nan
    return (alpha[0], beta[0], int(nk[0])) if one else (alpha, beta, nk)


def rolling_sum(M, k):
    """Unshifted rectangular accumulation along axis 0; first k-1 rows NaN."""
    M, one = _as2d(M)
    valid = np.isfinite(M)
    c = np.vstack([np.zeros((1, M.shape[1])), np.cumsum(np.where(valid, M, 0.), axis=0)])
    count = np.vstack([np.zeros((1, M.shape[1])), np.cumsum(valid, axis=0)])
    out = np.full(M.shape, np.nan)
    if len(M) >= k:
        out[k-1:] = np.where(count[k:] - count[:-k] == k, c[k:] - c[:-k], np.nan)
    return out[:, 0] if one else out


def spi(monthly, months, years, scale, calibration=None, zero_method="cdf_at_zero",
        eps=1e-10, min_positive=10):
    """Calendar-month-specific Gamma (L-moment) SPI with explicit zero mass.

    monthly : (T,) or (T, P) monthly totals in strict chronological order
    calibration : None -> full record (primary, as in the manuscript), or (y0, y1)
    zero_method : 'cdf_at_zero' H(0)=q  (primary) | 'midpoint' H(0)=q/2
    Returns SPI array and a parameter table (one row per calendar month and column).
    """
    M, one = _as2d(monthly)
    months, years = np.asarray(months), np.asarray(years)
    acc = rolling_sum(M, scale)
    out = np.full(acc.shape, np.nan)
    params = []
    for m in range(1, 13):
        idx = np.where(months == m)[0]
        idx = idx[np.isfinite(acc[idx]).any(axis=1)]
        cal = idx if calibration is None else idx[(years[idx] >= calibration[0]) & (years[idx] <= calibration[1])]
        C = acc[cal]
        finite = np.isfinite(C)
        n_cal = finite.sum(axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            q0 = np.where(n_cal > 0, ((C <= 0) & finite).sum(axis=0) / n_cal, np.nan)
        alpha, beta, npos = lmom_gamma_fit(np.where(C > 0, C, np.nan))
        enough = npos >= min_positive
        alpha = np.where(enough, alpha, np.nan)
        E = acc[idx]
        with np.errstate(invalid="ignore", divide="ignore"):
            G = special.gammainc(alpha[None, :], np.where(E > 0, E, 0) / beta[None, :])
            H = q0[None, :] + (1 - q0[None, :]) * G
            z0 = q0 if zero_method == "cdf_at_zero" else q0 / 2
            H = np.where(E <= 0, z0[None, :], H)
        H = np.where(np.isfinite(E) & np.isfinite(alpha)[None, :] & np.isfinite(beta)[None, :], H, np.nan)
        out[idx] = stats.norm.ppf(np.clip(H, eps, 1 - eps))
        params.append(pd.DataFrame(dict(calendar_month=m, column=np.arange(M.shape[1]),
                                        n_calibration=n_cal, n_positive=npos, p_zero=q0,
                                        gamma_shape_alpha=alpha, gamma_scale_beta=beta)))
    par = pd.concat(params, ignore_index=True)
    par.insert(0, "scale_months", scale)
    return (out[:, 0] if one else out), par


def gamma_gof_bootstrap(x, B=999, seed=2026):
    """KS and Anderson–Darling GOF for a gamma fitted by L-moments, with a
    parametric-bootstrap p-value that re-fits the parameters on every replicate
    (Lilliefors-type correction for estimated parameters)."""
    x = np.sort(np.asarray(x, float)[np.isfinite(x) & (np.asarray(x) > 0)])
    n = len(x)
    a, b, _ = lmom_gamma_fit(x)

    def stats_for(S):            # S: (n, K) sorted samples
        A, Bt, _ = lmom_gamma_fit(S)
        U = np.clip(special.gammainc(A[None, :], S / Bt[None, :]), 1e-12, 1 - 1e-12)
        i = np.arange(1, n + 1)[:, None]
        D = np.max(np.maximum(i / n - U, U - (i - 1) / n), axis=0)
        A2 = -n - np.mean((2 * i - 1) * (np.log(U) + np.log(1 - U[::-1])), axis=0)
        return D, A2

    D0, A0 = stats_for(x[:, None])
    rng = np.random.default_rng(seed)
    sims = np.sort(rng.gamma(a, b, size=(n, B)), axis=0)
    Ds, As = stats_for(sims)
    return dict(n_positive=n, gamma_shape_alpha=a, gamma_scale_beta=b,
                ks_D=D0[0], ks_p_boot=(1 + np.sum(Ds >= D0[0])) / (B + 1),
                ad_A2=A0[0], ad_p_boot=(1 + np.sum(As >= A0[0])) / (B + 1))


# ------------------------------------------------------- change points -----
def pettitt(x):
    x = np.asarray(x, float)
    n = len(x)
    r = stats.rankdata(x)
    k = np.arange(1, n + 1)
    U = 2 * np.cumsum(r) - k * (n + 1)
    tau = int(np.argmax(np.abs(U[:-1])))
    K = float(np.abs(U[tau]))
    return dict(K=K, tau_index=tau, p_approx=min(1.0, 2 * np.exp(-6 * K ** 2 / (n ** 3 + n ** 2))))


def buishand_range(x):
    x = np.asarray(x, float)
    n = len(x)
    S = np.cumsum(x - x.mean())[:-1]
    D = x.std()
    tau = int(np.argmax(np.abs(S)))
    return dict(Q=float(np.abs(S).max() / D), R=float((S.max() - min(S.min(), 0)) / D), tau_index=tau)


def snht(x):
    x = np.asarray(x, float)
    n = len(x)
    z = (x - x.mean()) / x.std(ddof=1)
    k = np.arange(1, n)
    c = np.cumsum(z)[:-1]
    T = k * (c / k) ** 2 + (n - k) * ((z.sum() - c) / (n - k)) ** 2
    tau = int(np.argmax(T))
    return dict(T0=float(T[tau]), tau_index=tau)


def _perm_matrix(x, B, rng):
    return np.asarray(x, float)[np.argsort(rng.random((B, len(x))), axis=1)]


def changepoint_battery(x, years, B=9999, seed=2026):
    """Pettitt, Buishand Q/R and SNHT with permutation p-values (exchangeability null)."""
    x = np.asarray(x, float)
    years = np.asarray(years)
    rng = np.random.default_rng(seed)
    Xp = _perm_matrix(x, B, rng)
    pt, bu, sn = pettitt(x), buishand_range(x), snht(x)
    Kp = np.array([pettitt(r)["K"] for r in Xp])
    Qp = np.array([buishand_range(r)["Q"] for r in Xp])
    Rp = np.array([buishand_range(r)["R"] for r in Xp])
    Tp = np.array([snht(r)["T0"] for r in Xp])
    P = lambda obs, sim: (1 + np.sum(sim >= obs)) / (B + 1)
    rows = [
        dict(test="Pettitt (rank-based)", statistic=pt["K"], change_year=int(years[pt["tau_index"]]),
             p_asymptotic=pt["p_approx"], p_permutation=P(pt["K"], Kp)),
        dict(test="Buishand range Q", statistic=bu["Q"], change_year=int(years[bu["tau_index"]]),
             p_asymptotic=np.nan, p_permutation=P(bu["Q"], Qp)),
        dict(test="Buishand range R", statistic=bu["R"], change_year=int(years[bu["tau_index"]]),
             p_asymptotic=np.nan, p_permutation=P(bu["R"], Rp)),
        dict(test="SNHT (Alexandersson)", statistic=sn["T0"], change_year=int(years[sn["tau_index"]]),
             p_asymptotic=np.nan, p_permutation=P(sn["T0"], Tp)),
    ]
    df = pd.DataFrame(rows)
    df["note"] = "change_year = last year of first segment; permutation null = exchangeable years (no trend, no step)"
    return df


def _ols(X, y):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    rss = float(np.sum((y - X @ beta) ** 2))
    return beta, rss


def structural_models(x, years, min_seg=5):
    """Compare constant, linear-trend, step, step+trend and broken-trend models
    by BIC/AICc. The break year is estimated by least squares and counted as a
    parameter. Distinguishes an abrupt shift from a gradual monotonic trend."""
    x = np.asarray(x, float)
    years = np.asarray(years)
    n = len(x)
    t = (years - years[0]).astype(float)
    one = np.ones(n)
    res = []

    def add(name, k, rss, brk=None):
        ll = -0.5 * n * (np.log(2 * np.pi * rss / n) + 1)
        aic = 2 * k - 2 * ll
        res.append(dict(model=name, n_params=k, rss=rss, break_year=brk,
                        AICc=aic + 2 * k * (k + 1) / (n - k - 1), BIC=k * np.log(n) - 2 * ll))

    add("Constant mean", 2, _ols(one[:, None], x)[1])
    add("Linear trend", 3, _ols(np.c_[one, t], x)[1])
    for name, design, k in [
        ("Step change (mean shift)", lambda s: np.c_[one, (t > t[s]).astype(float)], 4),
        ("Step change + common trend", lambda s: np.c_[one, t, (t > t[s]).astype(float)], 5),
        ("Broken (hinge) trend", lambda s: np.c_[one, t, np.maximum(t - t[s], 0)], 5),
    ]:
        best = min(((_ols(design(s), x)[1], s) for s in range(min_seg - 1, n - min_seg)), key=lambda z: z[0])
        add(name, k, best[0], int(years[best[1]]))
    df = pd.DataFrame(res)
    df["delta_BIC"] = df["BIC"] - df["BIC"].min()
    df["delta_AICc"] = df["AICc"] - df["AICc"].min()
    w = np.exp(-0.5 * df["delta_BIC"])
    df["BIC_weight"] = w / w.sum()
    return df.sort_values("BIC").reset_index(drop=True)


def changepoint_uncertainty(x, years, B=2000, L=None, seed=2026):
    """(i) Block-bootstrap distribution of the Pettitt change year under the
    fitted step model; (ii) how often Pettitt is 'significant' (p<0.05) when the
    data are generated by a pure linear trend with the observed residual
    structure. (ii) quantifies whether a gradual trend alone can produce the
    candidate change point."""
    x = np.asarray(x, float)
    years = np.asarray(years)
    n = len(x)
    L = L or max(2, int(round(n ** (1 / 3))))
    rng = np.random.default_rng(seed)
    tau = pettitt(x)["tau_index"]
    step_fit = np.where(np.arange(n) <= tau, x[:tau + 1].mean(), x[tau + 1:].mean())
    r_step = x - step_fit
    t = np.arange(n, dtype=float)
    b, a = sen_slope(x, t)
    trend_fit = a + b * t
    r_trend = x - trend_fit
    idx = _mbb_indices(n, B, L, rng)
    yrs_boot = np.array([years[pettitt(step_fit + r_step[ii])["tau_index"]] for ii in idx])
    p_trend_null = np.array([pettitt(trend_fit + r_trend[ii])["p_approx"] for ii in idx])
    return dict(
        bootstrap_change_years=yrs_boot,
        change_year_mode=int(pd.Series(yrs_boot).mode().iloc[0]),
        change_year_ci90=(int(np.quantile(yrs_boot, 0.05)), int(np.quantile(yrs_boot, 0.95))),
        change_year_ci95=(int(np.quantile(yrs_boot, 0.025)), int(np.quantile(yrs_boot, 0.975))),
        share_boot_within_pm2yr=float(np.mean(np.abs(yrs_boot - years[tau]) <= 2)),
        pettitt_rejection_rate_under_pure_trend=float(np.mean(p_trend_null < 0.05)),
        block_length=L, B=B)


def hodges_lehmann_shift(a, b, B=9999, seed=2026):
    """Median of pairwise differences (b - a) with a permutation p-value."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    hl = float(np.median(b[:, None] - a[None, :]))
    pooled = np.r_[a, b]
    rng = np.random.default_rng(seed)
    P = pooled[np.argsort(rng.random((B, len(pooled))), axis=1)]
    sims = np.array([np.median(r[len(a):][:, None] - r[:len(a)][None, :]) for r in P])
    return dict(hl_shift=hl, p_permutation=(1 + np.sum(np.abs(sims) >= abs(hl))) / (B + 1))


# --------------------------------------------- pentad dryness & concentration
def longest_run(D):
    """Longest run of True along axis 1 of a (Y, K, P) boolean array -> (Y, P)."""
    run = np.zeros((D.shape[0], D.shape[2]))
    best = run.copy()
    for k in range(D.shape[1]):
        run = np.where(D[:, k, :], run + 1, 0)
        best = np.maximum(best, run)
    return best


def calendar_percentiles(R, years, base=(1991, 2020)):
    """Mid-rank empirical percentile of every pentad against the same calendar
    pentad in the reference period (identical to the submitted definition).
    R: (Y, 72, P) -> (Y, 72, P)."""
    ref = (years >= base[0]) & (years <= base[1])
    Rref = R[ref]                                   # (Nref, 72, P)
    n = Rref.shape[0]
    out = np.empty(R.shape)
    for k in range(R.shape[1]):
        v = R[:, k, :][:, None, :]
        r = Rref[:, k, :][None, :, :]
        below = (r < v).sum(axis=1)
        equal = (r == v).sum(axis=1)
        out[:, k, :] = np.clip((below + 0.5 * equal + 0.5) / (n + 1.0), 1e-6, 1 - 1e-6)
    return out


def pentad_dryness(R, years, definition, base=(1991, 2020), months=None, pct=None):
    """Annual (or seasonal) dry-pentad count and longest dry run for one definition.

    definition keys:
      kind='percentile', q=0.20, min_clim_mm=None|float  (eligible calendar pentads only)
      kind='absolute',   thr_mm=2.0
    Pentads outside `months` or ineligible pentads are not dry and break runs.
    """
    Y = R.shape[0]
    pentad_month = np.repeat(np.arange(1, 13), 6)
    in_season = np.ones(72, bool) if months is None else np.isin(pentad_month, months)
    if definition["kind"] == "percentile":
        pct = calendar_percentiles(R, years, base) if pct is None else pct
        D = pct <= definition["q"]
        if definition.get("min_clim_mm") is not None:
            ref = (years >= base[0]) & (years <= base[1])
            clim = R[ref].mean(axis=0)              # (72, P)
            D &= (clim >= definition["min_clim_mm"])[None]
    else:
        D = R < definition["thr_mm"]
    D = D[:, in_season, :]
    return D.sum(axis=1).astype(float), longest_run(D)


def concentration_metrics(R, wet_mm=5.0):
    """Precisely defined annual concentration metrics from (Y, 72, P) pentads."""
    total = R.sum(axis=1)
    desc = -np.sort(-R, axis=1)
    asc = np.sort(R, axis=1)
    n = R.shape[1]
    i = np.arange(1, n + 1)[None, :, None]
    with np.errstate(invalid="ignore", divide="ignore"):
        gini = (2 * (i * asc).sum(axis=1)) / (n * total) - (n + 1) / n
        mon = R.reshape(R.shape[0], 12, 6, -1).sum(axis=2)
        pci = 100 * (mon ** 2).sum(axis=1) / total ** 2
        wet = R >= wet_mm
        n_wet = wet.sum(axis=1).astype(float)
        p_wet = np.where(wet, R, 0).sum(axis=1)
        return dict(
            annual_total_mm=total,
            top10_share_pct=100 * desc[:, :10].sum(axis=1) / total,
            top5_share_pct=100 * desc[:, :5].sum(axis=1) / total,
            max_pentad_mm=desc[:, 0],
            n_wet_pentads_ge5=n_wet,
            wet_pentad_intensity_mm=np.where(n_wet > 0, p_wet / n_wet, np.nan),
            rain_from_wet_pentads_mm=p_wet,
            rain_from_sub5mm_pentads_mm=total - p_wet,
            wet_pentad_share_pct=100 * p_wet / total,
            gini_pentad=gini,
            pci_monthly=pci,
        )


print("Statistical methods loaded.")

In [ ]:
# =============================================================================
# 3. Method self-tests (each checks a known answer; the notebook stops on failure)
# =============================================================================
_rng = np.random.default_rng(1)
# Benjamini–Hochberg against a hand-computed example
assert np.allclose(bh_adjust([0.01, 0.04, 0.03, 0.20]), [0.04, 0.16 / 3, 0.16 / 3, 0.20])
# Sen slope of an exact line; Mann–Kendall detects a strong monotonic trend, not white noise
assert np.isclose(sen_slope(2.5 * np.arange(30.0) + 7)[0], 2.5)
assert mk_hamed_rao(np.arange(40.0) + _rng.normal(0, 3, 40))["p_hr"] < 1e-6
_null_p = [mk_hamed_rao(_rng.normal(size=45))["p_hr"] for _ in range(400)]
assert 0.02 < np.mean(np.array(_null_p) < 0.05) < 0.09, "MK size under the null is off"
# Rolling sums propagate gaps instead of bridging them
assert np.isnan(rolling_sum(np.array([1.0, np.nan, 2.0, 3.0]), 2)[2])
assert np.isclose(rolling_sum(np.array([1.0, np.nan, 2.0, 3.0]), 2)[3], 5.0)
# L-moment gamma recovers known parameters
_a, _b, _ = lmom_gamma_fit(_rng.gamma(2.0, 30.0, 200000))
assert abs(_a - 2.0) < 0.03 and abs(_b - 30.0) < 0.6
# SPI of gamma data is ~N(0,1)
_g = _rng.gamma(1.5, 20.0, 12 * 200)
_s, _ = spi(_g, np.tile(np.arange(1, 13), 200), np.repeat(np.arange(200), 12), 1)
assert abs(np.nanmean(_s)) < 0.05 and abs(np.nanstd(_s) - 1) < 0.05
# Pettitt locates an obvious step
assert pettitt(np.r_[np.zeros(20), np.ones(20) * 5] + _rng.normal(0, 0.3, 40))["tau_index"] == 19
print("All method self-tests passed.")

## 4. Data acquisition
Runs only when a cache file is missing and `ALLOW_EARTH_ENGINE = True`; otherwise the archived inputs are used unchanged.

In [ ]:
# =============================================================================
# 4. Data acquisition from Google Earth Engine (runs only if a cache is missing
#    and ALLOW_EARTH_ENGINE = True)
# -----------------------------------------------------------------------------
# (a) Regional pentad series: area-weighted CHIRPS v2 mean over the dissolved
#     Cholistan boundary, Σ(P·A)/ΣA with ee.Image.pixelArea at the native 5,566 m
#     scale, requested one year (72 pentads) at a time.
# (b) Native-grid pentad cubes (CHIRPS v2 and v3): one 72-band computePixels
#     request per year on the CHIRPS 0.05° lattice (Section 9).
# (c) Monthly regional series for PERSIANN-CDR, ERA5-Land and IMERG V07 (Section 10).
# =============================================================================
_EE_READY = False


def ee_init():
    global _EE_READY
    if not ALLOW_EARTH_ENGINE:
        raise RuntimeError("A required cache is missing. Set ALLOW_EARTH_ENGINE = True to rebuild it from Earth Engine.")
    import ee
    if not _EE_READY:
        ee.Initialize(project=EE_PROJECT_ID)
        _EE_READY = True
    return ee


def ee_retry(fn, label, retries=4):
    for attempt in range(retries):
        try:
            return fn()
        except Exception as exc:                       # quota / transient service errors
            if attempt == retries - 1:
                raise
            print(f"  retry {attempt + 1} for {label}: {str(exc)[:100]}")
            time.sleep(5 * 2 ** attempt)


def extract_regional_pentads():
    ee = ee_init()
    geom = ee.FeatureCollection(EE_BOUNDARY_ASSET).geometry()

    def feat(img):
        img = ee.Image(img)
        rain = img.select("precipitation")
        area = ee.Image.pixelArea().updateMask(rain.mask()).rename("a")
        s = ee.Image.cat([rain.multiply(area).rename("pa"), area]).reduceRegion(
            ee.Reducer.sum(), geom, scale=5566, maxPixels=1e9, tileScale=8, bestEffort=False)
        return ee.Feature(None, {"date": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"),
                                 "rainfall_mm": ee.Number(s.get("pa")).divide(ee.Number(s.get("a")))})
    frames = []
    for y in YEARS:
        col = ee.ImageCollection(CHIRPS_V2_PENTAD).filterDate(f"{y}-01-01", f"{y + 1}-01-01").select("precipitation")
        fc = ee.FeatureCollection(col.map(feat))
        df = ee_retry(lambda: ee.data.computeFeatures({"expression": fc, "fileFormat": "PANDAS_DATAFRAME"}), str(y))
        frames.append(df.drop(columns=[c for c in ("geo", ".geo") if c in df.columns]))
        print(y, end=" ")
    out = pd.concat(frames, ignore_index=True).sort_values("date")
    if len(out) != len(YEARS) * 72:
        raise ValueError(f"Expected {len(YEARS) * 72} pentads, received {len(out)}")
    return out


if not PENTAD_CACHE_CSV.exists():
    extract_regional_pentads().to_csv(PENTAD_CACHE_CSV, index=False)
print("Regional pentad series:", PENTAD_CACHE_CSV.name)

## 5. Input quality control and rainfall regime
The archived series must contain exactly six pentads per month and 72 per year, with no gaps or negative values, and must reproduce the values reported in the manuscript; the notebook stops otherwise.

In [ ]:
# =============================================================================
# 5. Input quality control, reproduction of submitted values, climatology
# =============================================================================
pent = pd.read_csv(PENTAD_CACHE_CSV, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
pent["year"], pent["month"] = pent.date.dt.year, pent.date.dt.month
pent["pentad_in_month"] = pent.groupby(["year", "month"]).cumcount() + 1
assert not pent.date.duplicated().any(), "duplicate pentads"
assert (pent.groupby(["year", "month"]).size() == 6).all(), "each month must have six pentads"
assert pent.year.min() == START_YEAR and pent.year.max() == END_YEAR and len(pent) == len(YEARS) * 72
assert pent.rainfall_mm.notna().all() and (pent.rainfall_mm >= 0).all()

R_reg = pent.rainfall_mm.to_numpy().reshape(len(YEARS), 72, 1)              # (year, pentad, 1)
monthly = pent.groupby(["year", "month"], as_index=False).rainfall_mm.sum()
M_reg, M_months, M_years = monthly.rainfall_mm.to_numpy(), monthly.month.to_numpy(), monthly.year.to_numpy()
M_dates = pd.to_datetime(dict(year=M_years, month=M_months, day=1))
annual = R_reg[:, :, 0].sum(axis=1)
seasonal = {s: monthly[monthly.month.isin(m)].groupby("year").rainfall_mm.sum().to_numpy() for s, m in SEASONS.items()}
ref = (YEARS >= NORMAL[0]) & (YEARS <= NORMAL[1])

# Reproduction of values reported in the submitted manuscript
checks = [("1991–2020 mean annual rainfall (mm)", annual[ref].mean(), 179.9, 0.05),
          ("Driest year total, 2002 (mm)", annual[YEARS == 2002][0], 69.1, 0.05),
          ("Wettest year total, 2022 (mm)", annual[YEARS == 2022][0], 437.3, 0.05),
          ("Annual Sen slope (mm/yr)", mk_hamed_rao(annual)["sen_slope"], 3.05, 0.005),
          ("JJAS Sen slope (mm/yr)", mk_hamed_rao(seasonal["JJAS"])["sen_slope"], 2.48, 0.005),
          ("Pettitt change year", YEARS[pettitt(annual)["tau_index"]], 2009, 0)]
reproduction = pd.DataFrame([dict(item=i, reproduced=round(float(v), 3), manuscript=m, match=abs(float(v) - m) <= tol)
                             for i, v, m, tol in checks])
display(reproduction)
assert reproduction.match.all(), "Archived input does not reproduce the submitted values"
save_table(reproduction, "Table_S01_reproduction_of_reported_values", "Supplementary",
           "Values reported in the submitted manuscript re-derived from the archived regional CHIRPS v2 series.")

clim = monthly[monthly.year.between(*NORMAL)].groupby("month").rainfall_mm.agg(["mean", "median", "std"]).reset_index()
clim["share_of_annual_pct"] = 100 * clim["mean"] / clim["mean"].sum()
save_table(clim.round(3), "Table_S02_monthly_climatology_1991_2020", "Supplementary",
           "Monthly rainfall climatology, 1991–2020 (area-weighted CHIRPS v2).")
annual_table = pd.DataFrame(dict(year=YEARS, annual_mm=annual, JJAS_mm=seasonal["JJAS"], JAS_mm=seasonal["JAS"],
                                 annual_departure_pct=100 * (annual / annual[ref].mean() - 1)))
save_table(annual_table.round(2), "Table_S03_annual_and_monsoon_rainfall", "Supplementary",
           "Annual, JJAS and JAS rainfall totals and annual departure from the 1991–2020 mean.")

# ---- Figure: rainfall regime ----------------------------------------------------------------
fig = new_figure("double", 90)
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.9])
ax = fig.add_subplot(gs[0])
ax.bar(clim.month, clim["mean"], color=PAL["purple"], width=0.72)
ax.errorbar(clim.month, clim["mean"], yerr=clim["std"], fmt="none", ecolor=PAL["muted"], elinewidth=0.6, capsize=1.5)
ax.set(xticks=range(1, 13), xticklabels=list("JFMAMJJASOND"), ylabel="Rainfall (mm month$^{-1}$)", ylim=(0, None))
ax.set_title("Monthly climatology, 1991–2020 (mean ± SD)")
panel_label(ax, "a")
ax = fig.add_subplot(gs[1])
ax.plot(YEARS, annual, color=PAL["purple_dark"], marker="o", ms=2.2, label="Annual")
ax.plot(YEARS, seasonal["JJAS"], color=PAL["purple_mid"], marker="o", ms=2.2, label="JJAS")
ax.axhline(annual[ref].mean(), color=PAL["muted"], lw=0.6, ls="--", label=f"1991–2020 annual mean ({annual[ref].mean():.0f} mm)")
ax.set(xlabel="Year", ylabel="Rainfall (mm)", xlim=(YEARS[0] - 1, YEARS[-1] + 1), ylim=(0, None))
ax.legend(loc="upper left")
ax.set_title("Annual and monsoon (JJAS) totals")
panel_label(ax, "b")
save_figure(fig, "Fig02_rainfall_regime", "Main",
            f"Rainfall regime of the Cholistan Desert from area-weighted CHIRPS v2 pentads. (a) Monthly climatology "
            f"1991–2020 (bars: mean; whiskers: ±1 SD); JJAS supplies {clim.loc[clim.month.between(6, 9), 'share_of_annual_pct'].sum():.0f}% "
            f"of the annual mean. (b) Annual and JJAS totals, {START_YEAR}–{END_YEAR}; dashed line: 1991–2020 annual mean.")
print("Saved Fig02.")

## 6. Standardized Precipitation Index
* Monthly totals are sums of the six CHIRPS pentads. Accumulations are unshifted k-month running sums (k = 1, 3, 6, 12); the first k−1 months are undefined.
* For each ending calendar month, a two-parameter gamma distribution is fitted to positive accumulations by L-moments (Hosking 1990), calibrated on the full 1981–2025 record. Zeros enter as a mixed distribution H(x) = q + (1 − q)G(x) with H(0) = q; SPI = Φ⁻¹(H), with probabilities clipped to [10⁻¹⁰, 1 − 10⁻¹⁰].
* Goodness of fit is tested with Kolmogorov–Smirnov and Anderson–Darling statistics, using parametric-bootstrap p-values that re-estimate the parameters in each replicate. Sensitivity is tested for 1991–2020 calibration and for H(0) = q/2.
* **Events.** A *strict run* is a sequence of consecutive months with SPI ≤ −1 at one scale. An *event envelope* is a negative-SPI sequence that reaches −1. Major episodes (Table 1) merge severe or persistent strict runs across SPI-3/6/12, as described in the table caption.
* **Calibration caveat.** Calibrating on the full record, which contains a wetter post-2009 period, places the record's SPI drought months mostly before 2010. The 1991–2020 calibration is reported in Table S6.

In [ ]:
# =============================================================================
# 6. Standardized Precipitation Index: fit, goodness of fit, sensitivity, events
# =============================================================================
spi_series, spi_params, gof_rows = {}, [], []
for k in SPI_SCALES:
    s, par = spi(M_reg, M_months, M_years, k, calibration=SPI_PRIMARY_CALIBRATION)
    s91, _ = spi(M_reg, M_months, M_years, k, calibration=SPI_SENSITIVITY_CALIBRATION)
    smid, _ = spi(M_reg, M_months, M_years, k, calibration=SPI_PRIMARY_CALIBRATION, zero_method="midpoint")
    spi_series[k] = dict(primary=s, cal_1991_2020=s91, zero_midpoint=smid)
    spi_params.append(par.drop(columns="column"))
    acc = rolling_sum(M_reg, k)
    for m in range(1, 13):
        sel = (M_months == m) & np.isfinite(acc)
        g = gamma_gof_bootstrap(acc[sel], B=999, seed=SEED + 100 * k + m)
        sw = stats.shapiro(s[sel & np.isfinite(s)])
        gof_rows.append(dict(scale=f"SPI-{k}", calendar_month=m, **g, spi_mean=np.nanmean(s[sel]),
                             spi_sd=np.nanstd(s[sel], ddof=1), shapiro_W=sw.statistic, shapiro_p=sw.pvalue))
spi_params = pd.concat(spi_params, ignore_index=True)
spi_gof = pd.DataFrame(gof_rows)
for c in ("ks_p_boot", "ad_p_boot", "shapiro_p"):
    spi_gof[c.replace("_boot", "") .replace("_p", "_q_bh")] = spi_gof.groupby("scale")[c].transform(bh_adjust)
gof_summary = spi_gof.groupby("scale", sort=False).agg(
    ks_reject_q05=("ks_q_bh", lambda q: int((q < 0.05).sum())), ad_reject_q05=("ad_q_bh", lambda q: int((q < 0.05).sum())),
    ad_rejected_months=("ad_q_bh", lambda q: ",".join(str(m) for m, v in zip(range(1, 13), q) if v < 0.05) or "none"),
    shapiro_reject_q05=("shapiro_q_bh", lambda q: int((q < 0.05).sum())), min_positive_n=("n_positive", "min")).reset_index()
display(gof_summary)
save_table(spi_params.round(5), "Table_S04_SPI_gamma_parameters", "Supplementary",
           "Gamma (L-moment) parameters and zero probability by SPI scale and ending calendar month; full-record calibration.")
save_table(spi_gof.round(4), "Table_S05_SPI_goodness_of_fit", "Supplementary",
           "Goodness of fit of the gamma model: Kolmogorov–Smirnov and Anderson–Darling with parametric-bootstrap p-values "
           "(parameters re-estimated in 999 replicates), Shapiro–Wilk of SPI; BH-adjusted across the 12 months of each scale.")

# Implementation check against the submitted SPI file, when present
_sub = CSV_IN / "spi_gamma_primary.csv"
if _sub.exists():
    _s = pd.read_csv(_sub, parse_dates=["date"])
    _d = max(np.nanmax(np.abs(_s[_s.scale_months == k].sort_values("date").spi.to_numpy() - spi_series[k]["primary"]))
             for k in SPI_SCALES)
    print(f"Maximum |SPI difference| vs submitted lmoments3 implementation: {_d:.2e}")
    assert _d < 1e-6

sens = []
for k in SPI_SCALES:
    p = spi_series[k]["primary"]
    for lab, nice in (("cal_1991_2020", "Calibration 1991–2020"), ("zero_midpoint", "Zero mass at q/2")):
        a = spi_series[k][lab]
        ok = np.isfinite(p) & np.isfinite(a)
        sens.append(dict(scale=f"SPI-{k}", alternative=nice, pearson_r=np.corrcoef(p[ok], a[ok])[0, 1],
                         mean_abs_diff=np.mean(np.abs(a[ok] - p[ok])),
                         drought_class_disagreement_pct=100 * np.mean((p[ok] <= -1) != (a[ok] <= -1)),
                         drought_months_primary=int((p[ok] <= -1).sum()), drought_months_alternative=int((a[ok] <= -1).sum())))
save_table(pd.DataFrame(sens).round(4), "Table_S06_SPI_calibration_sensitivity", "Supplementary",
           "SPI sensitivity to the calibration period and to the treatment of zero accumulations.")

# ---- Drought events ----------------------------------------------------------------------
def longest_true_run_months(values):
    values = np.asarray(values, dtype=bool)
    best = run = 0
    for v in values:
        run = run + 1 if v else 0
        best = max(best, run)
    return int(best)

def extract_spi_event_envelopes(dates, spi_values, scale_label):
    # Classical event envelope:
    # starts at SPI < 0, must reach SPI <= -1, ends before SPI becomes non-negative.
    dates = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    spi = np.asarray(spi_values, dtype=float)
    negative = np.isfinite(spi) & (spi < 0)

    events = []
    i = 0
    event_id = 0

    while i < len(spi):
        if not negative[i]:
            i += 1
            continue

        start = i
        while i + 1 < len(spi) and negative[i + 1]:
            i += 1
        end = i

        idx = np.arange(start, end + 1)
        x = spi[idx]
        core_local = np.where(x <= -1.0)[0]

        if len(core_local):
            event_id += 1
            first_core_local = int(core_local[0])
            last_core_local = int(core_local[-1])
            core_mask = x <= -1.0

            peak_local = int(np.nanargmin(x))
            peak = float(np.nanmin(x))
            category = (
                "Extreme" if peak <= -2.0
                else "Severe" if peak <= -1.5
                else "Moderate"
            )

            envelope_duration = int(len(idx))
            threshold_months = int(core_mask.sum())

            events.append({
                "event_id": event_id,
                "scale": scale_label,
                "envelope_start_date": dates.iloc[start],
                "threshold_onset_date": dates.iloc[int(idx[first_core_local])],
                "threshold_last_date": dates.iloc[int(idx[last_core_local])],
                "envelope_end_date": dates.iloc[end],
                "envelope_duration_months": envelope_duration,
                "threshold_months_spi_le_minus1": threshold_months,
                "longest_consecutive_threshold_run_months": longest_true_run_months(core_mask),
                "lead_in_negative_months": first_core_local,
                "recovery_tail_negative_months": envelope_duration - 1 - last_core_local,
                "envelope_cumulative_negative_spi": float(-np.nansum(x)),
                "threshold_cumulative_negative_spi": float(-np.nansum(x[core_mask])),
                "envelope_mean_negative_intensity": float(-np.nanmean(x)),
                "threshold_mean_intensity": float(-np.nanmean(x[core_mask])),
                "peak_spi": peak,
                "peak_date": dates.iloc[int(idx[peak_local])],
                "peak_category": category,

            })

        i += 1

    return pd.DataFrame(events)

def extract_strict_spi_threshold_runs(dates, spi_values, scale_label, threshold=-1.0):
    dates = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    spi = np.asarray(spi_values, dtype=float)
    flag = np.isfinite(spi) & (spi <= threshold)

    rows = []
    i = 0
    event_id = 0

    while i < len(spi):
        if not flag[i]:
            i += 1
            continue

        start = i
        while i + 1 < len(spi) and flag[i + 1]:
            i += 1
        end = i

        idx = np.arange(start, end + 1)
        x = spi[idx]
        event_id += 1

        peak_local = int(np.nanargmin(x))
        peak = float(np.nanmin(x))
        category = (
            "Extreme" if peak <= -2.0
            else "Severe" if peak <= -1.5
            else "Moderate"
        )

        rows.append({
            "event_id": event_id,
            "scale": scale_label,
            "threshold": threshold,
            "start_date": dates.iloc[start],
            "end_date": dates.iloc[end],
            "duration_months": int(len(idx)),
            "cumulative_negative_spi": float(-np.nansum(x)),
            "mean_intensity": float(-np.nanmean(x)),
            "peak_spi": peak,
            "peak_date": dates.iloc[int(idx[peak_local])],
            "peak_category": category,
        })

        i += 1

    return pd.DataFrame(rows)


def build_cross_scale_major_drought_table(events):
    e = events.copy()
    e = e[e["scale"].isin(["SPI-3", "SPI-6", "SPI-12"])].copy()
    for c in ["start_date", "end_date", "peak_date"]:
        e[c] = pd.to_datetime(e[c])

    # Candidate major events: severe/extreme peak or persistent duration >=6 months.
    e = e[(e["peak_spi"] <= -1.5) | (e["duration_months"] >= 6)].sort_values("start_date")
    if e.empty:
        return pd.DataFrame()

    clusters = []
    current = []
    current_end = None
    tolerance = pd.Timedelta(days=62)

    for _, row in e.iterrows():
        if not current:
            current = [row]
            current_end = row["end_date"]
            continue
        if row["start_date"] <= current_end + tolerance:
            current.append(row)
            current_end = max(current_end, row["end_date"])
        else:
            clusters.append(current)
            current = [row]
            current_end = row["end_date"]
    if current:
        clusters.append(current)

    rows = []
    scale_order = {"SPI-3": 3, "SPI-6": 6, "SPI-12": 12}
    for cluster in clusters:
        c = pd.DataFrame(cluster)
        strongest = c.loc[c["peak_spi"].idxmin()]
        scales = sorted(c["scale"].unique(), key=lambda s: scale_order[s])
        scale_count = len(scales)
        max_duration = int(c["duration_months"].max())
        min_spi = float(strongest["peak_spi"])

        # Keep cross-scale or particularly strong/persistent episodes.
        if not (scale_count >= 2 or min_spi <= -2.0 or max_duration >= 6):
            continue

        start = c["start_date"].min()
        end = c["end_date"].max()
        rows.append({
            "Episode": f"{start:%b %Y}–{end:%b %Y}",
            "SPI scales involved": ", ".join(scales),
            "Strongest SPI": round(min_spi, 2),
            "Peak month": pd.Timestamp(strongest["peak_date"]).strftime("%b %Y"),
            "Strongest class": strongest["peak_category"],
            "Maximum strict-run duration (months)": max_duration,
        })
    return pd.DataFrame(rows)


spi_primary = pd.concat([pd.DataFrame(dict(date=M_dates, scale_months=k, spi=v["primary"])) for k, v in spi_series.items()],
                        ignore_index=True)
envelopes, strict_runs = [], []
for k in SPI_SCALES:
    g = spi_primary[spi_primary.scale_months == k].sort_values("date")
    envelopes.append(extract_spi_event_envelopes(g.date, g.spi, f"SPI-{k}"))
    strict_runs.append(extract_strict_spi_threshold_runs(g.date, g.spi, f"SPI-{k}"))
envelopes, strict_runs = pd.concat(envelopes, ignore_index=True), pd.concat(strict_runs, ignore_index=True)
event_summary = pd.DataFrame([dict(
    scale=f"SPI-{k}", valid_months=int(np.isfinite(spi_series[k]["primary"]).sum()),
    drought_months=int((spi_series[k]["primary"] <= -1).sum()),
    drought_month_frequency_pct=100 * np.nanmean(spi_series[k]["primary"][np.isfinite(spi_series[k]["primary"])] <= -1),
    event_envelopes=int((envelopes.scale == f"SPI-{k}").sum()), strict_runs=int((strict_runs.scale == f"SPI-{k}").sum()),
    longest_strict_run_months=int(strict_runs.loc[strict_runs.scale == f"SPI-{k}", "duration_months"].max()),
    minimum_spi=np.nanmin(spi_series[k]["primary"]),
    minimum_month=M_dates[np.nanargmin(spi_series[k]["primary"])].strftime("%Y-%m")) for k in SPI_SCALES])
display(event_summary.round(2))
table1 = build_cross_scale_major_drought_table(strict_runs)
display(table1)
save_table(table1, "Table_1_major_meteorological_drought_episodes", "Main",
           "Major meteorological drought episodes, 1981–2025. Strict runs (consecutive months with SPI ≤ −1) at SPI-3, -6 and "
           "-12 with peak ≤ −1.5 or duration ≥ 6 months were merged across scales when separated by ≤ 62 days; episodes "
           "involving ≥ 2 scales, peaking ≤ −2, or lasting ≥ 6 months are listed. SPI: gamma, L-moments, full-record calibration.")
save_table(event_summary.round(3), "Table_S07_SPI_event_summary", "Supplementary",
           "Drought-month frequency, event counts and extremes by SPI scale.")
save_table(strict_runs.round(3), "Table_S08_SPI_strict_threshold_runs", "Supplementary",
           "All runs of consecutive months with SPI ≤ −1, by scale.")
save_table(envelopes.round(3), "Table_S09_SPI_event_envelopes", "Supplementary",
           "Event envelopes: negative-SPI sequences that reach SPI ≤ −1 (onset at first SPI < 0, end before SPI ≥ 0).")
save_table(spi_primary.round(4), "Table_S10_SPI_timeseries", "Supplementary", "Monthly SPI-1, -3, -6 and -12 (primary formulation).")

# ---- Figure: SPI time series ---------------------------------------------------------------------
import matplotlib.dates as mdates
fig = new_figure("double", 132)
axes = fig.subplots(3, 1, sharex=True, sharey=True)
for ax, k, letter in zip(axes, SPI_SCALES_MAIN, "abc"):
    s = spi_series[k]["primary"]
    ax.fill_between(M_dates, 0, s, where=s < 0, color=PAL["purple"], lw=0, interpolate=True)
    ax.fill_between(M_dates, 0, s, where=s >= 0, color=PAL["grey_light"], lw=0, interpolate=True)
    for thr, ls in ((-1, "--"), (-1.5, ":"), (-2, ":")):
        ax.axhline(thr, color=PAL["muted"], lw=0.5, ls=ls)
    ax.axhline(0, color=PAL["ink"], lw=0.4)
    ax.set_ylabel(f"SPI-{k}")
    ax.set_ylim(-3.6, 3.6)
    panel_label(ax, letter)
axes[-1].xaxis.set_major_locator(mdates.YearLocator(5))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[-1].set_xlim(M_dates.iloc[0], M_dates.iloc[-1])
save_figure(fig, "Fig03_SPI_3_6_12", "Main",
            "Standardized Precipitation Index at (a) 3, (b) 6 and (c) 12 months, 1981–2025 (gamma distribution, "
            "L-moment estimation by ending calendar month, full-record calibration). Purple: SPI < 0. Dashed: moderate "
            "drought threshold (−1); dotted: severe (−1.5) and extreme (−2).")
print("Saved Fig03 and SPI tables.")

## 7. Trends and change points
Trends use the Hamed–Rao (1998) modified Mann–Kendall test with Sen slopes. Each trend row also reports:
* a 95 % moving-block-bootstrap CI (block length n^(1/3))
* the lag-1 autocorrelation of the detrended series and the effective sample size
* unadjusted and trend-free pre-whitening p-values
* Benjamini–Hochberg q within each family

Change points are tested with Pettitt, Buishand range and SNHT, using permutation p-values. Abrupt and gradual change are compared with five structural models by BIC. The change-year uncertainty and the Pettitt test's behaviour under a pure trend are estimated by block bootstrap.

In [ ]:
# =============================================================================
# 7. Trends with serial-dependence diagnostics; change point vs gradual trend
# =============================================================================
rain_series = {"Annual rainfall": annual, "JJAS rainfall": seasonal["JJAS"], "JAS rainfall": seasonal["JAS"]}
rain_trends = trend_family([trend_row(x, YEARS, k, "mm/yr", B=N_BOOT, seed=SEED) for k, x in rain_series.items()],
                           "Rainfall totals")
burden_rows = []
for k in SPI_SCALES:
    s = spi_series[k]["primary"]
    yy = pd.DataFrame(dict(year=M_years, d=(s <= -1).astype(float), ok=np.isfinite(s))).groupby("year").agg(d=("d", "sum"), ok=("ok", "sum"))
    yy = yy[yy.ok == 12]
    burden_rows.append(trend_row(yy.d.to_numpy(), yy.index.to_numpy(), f"SPI-{k} drought months per year", "months/yr", B=N_BOOT))
    smin = pd.Series(s).groupby(M_years).min()
    smin = smin[yy.index]
    burden_rows.append(trend_row(smin.to_numpy(), smin.index.to_numpy(), f"SPI-{k} annual minimum", "SPI units/yr", B=N_BOOT))
burden_trends = trend_family(burden_rows, "SPI drought burden")
trends_all = pd.concat([rain_trends, burden_trends], ignore_index=True)
save_table(trends_all.round(5), "Table_S11_trends_with_autocorrelation_diagnostics", "Supplementary",
           "Hamed–Rao modified Mann–Kendall tests with Sen slopes (95% moving-block bootstrap CI, block length n^(1/3)), "
           "lag-1 autocorrelation of the detrended series, variance factor n/n*, effective sample size n*, unadjusted MK and "
           "trend-free pre-whitening p-values; BH q within family. For drought-month counts Sen slopes are degenerate "
           "(mostly zero); use Kendall's tau and p.")
display(trends_all[["series", "sen_slope_per_year", "sen_ci95_low", "sen_ci95_high", "kendall_tau", "lag1_acf_detrended",
                    "n_effective", "p_hr", "p_tfpw", "q_bh_within_family"]].round(3))

# ---- Change-point tests, structural models, uncertainty ------------------------------------
cp_tests = []
for name, x in rain_series.items():
    t = changepoint_battery(x, YEARS, B=N_PERM, seed=SEED)
    t.insert(0, "series", name)
    cp_tests.append(t)
cp_tests = pd.concat(cp_tests, ignore_index=True)
cp_tests["q_bh_permutation"] = bh_adjust(cp_tests["p_permutation"])
cp_models = structural_models(annual, YEARS)
cp_unc = changepoint_uncertainty(annual, YEARS, B=N_BOOT, seed=SEED)
display(cp_models.round(3))
print({k: v for k, v in cp_unc.items() if k != "bootstrap_change_years"})
save_table(cp_tests.drop(columns="note").round(5), "Table_S12_changepoint_tests", "Supplementary",
           "Pettitt, Buishand range (Q, R) and SNHT tests with permutation p-values (9,999 permutations); the change year is "
           "the last year of the first segment.")
save_table(cp_models.round(3), "Table_S13_step_versus_trend_models", "Supplementary",
           "Annual rainfall: constant, linear-trend, step, step + trend and hinge-trend models compared by BIC and AICc "
           "(break year estimated by least squares and counted as a parameter).")
save_table(pd.DataFrame([{k: v for k, v in cp_unc.items() if k != "bootstrap_change_years"}]), "Table_S14_changepoint_uncertainty",
           "Supplementary", "Block-bootstrap distribution of the Pettitt change year under the fitted step model, and the "
           "rejection rate of the Pettitt test when data are generated by a pure linear trend.")

sub_rows = []
for pname, (y0, y1) in SUBPERIODS.items():
    sel = (YEARS >= y0) & (YEARS <= y1)
    for name in ("Annual rainfall", "JJAS rainfall"):
        r = trend_row(rain_series[name][sel], YEARS[sel], name, "mm/yr", B=N_BOOT, seed=SEED)
        r["period"] = pname
        sub_rows.append(r)
for name in ("Annual rainfall", "JJAS rainfall"):
    x = rain_series[name]
    resid = x - np.where(YEARS <= CHANGE_YEAR, x[YEARS <= CHANGE_YEAR].mean(), x[YEARS > CHANGE_YEAR].mean())
    r = trend_row(resid, YEARS, name, "mm/yr", B=N_BOOT, seed=SEED)
    r["period"] = f"1981–2025, mean shift at {CHANGE_YEAR} removed"
    sub_rows.append(r)
subperiods = pd.DataFrame(sub_rows)
subperiods["q_bh_all"] = bh_adjust(subperiods["p_hr"])
shift = pd.DataFrame([dict(series=n, mean_1981_2009=x[YEARS <= CHANGE_YEAR].mean(), mean_2010_2025=x[YEARS > CHANGE_YEAR].mean(),
                           **hodges_lehmann_shift(x[YEARS <= CHANGE_YEAR], x[YEARS > CHANGE_YEAR], B=N_PERM, seed=SEED))
                      for n, x in rain_series.items()])
display(subperiods[["period", "series", "n", "sen_slope_per_year", "sen_ci95_low", "sen_ci95_high", "p_hr"]].round(3))
save_table(subperiods.round(4), "Table_S15_subperiod_trends", "Supplementary",
           "Trend sensitivity to the analysis window and after removing the mean shift at the candidate change year.")
save_table(shift.round(4), "Table_S16_segment_shift", "Supplementary",
           "Segment means and Hodges–Lehmann shift (2010–2025 minus 1981–2009) with permutation p-values.")
primary_trends = trends_all[trends_all.series.isin(["Annual rainfall", "JJAS rainfall", "SPI-3 annual minimum",
                                                     "SPI-6 annual minimum", "SPI-12 annual minimum",
                                                     "SPI-12 drought months per year"])]

# ---- Figure: abrupt shift or gradual trend ------------------------------------------------------
fig = new_figure("double", 128)
gs = fig.add_gridspec(2, 2, height_ratios=[1.3, 1])
ax = fig.add_subplot(gs[0, :])
ax.bar(YEARS, annual, color=PAL["purple_light"], width=0.8)
pre, post = YEARS <= CHANGE_YEAR, YEARS > CHANGE_YEAR
ax.hlines([annual[pre].mean(), annual[post].mean()], [YEARS[pre][0] - .4, YEARS[post][0] - .4],
          [YEARS[pre][-1] + .4, YEARS[post][-1] + .4], color=PAL["purple_dark"], lw=1.3,
          label=f"Segment means (1981–{CHANGE_YEAR}, {CHANGE_YEAR + 1}–2025)")
b, a = sen_slope(annual)
ax.plot(YEARS, a + b * np.arange(len(YEARS)), color=PAL["purple"], ls="--", lw=1, label="Sen trend 1981–2025")
hinge = int(cp_models.set_index("model").loc["Broken (hinge) trend", "break_year"])
t = (YEARS - YEARS[0]).astype(float)
Xh = np.c_[np.ones_like(t), t, np.maximum(t - (hinge - YEARS[0]), 0)]
ax.plot(YEARS, Xh @ np.linalg.lstsq(Xh, annual, rcond=None)[0], color=PAL["ink"], lw=1, label=f"Hinge model (break {hinge})")
lo, hi = cp_unc["change_year_ci95"]
ax.axvspan(lo - .5, hi + .5, color=PAL["grey"], alpha=0.2, lw=0, label="95% bootstrap range of change year")
ax.set(ylabel="Annual rainfall (mm)", xlim=(YEARS[0] - 1, YEARS[-1] + 1))
ax.legend(ncol=2, loc="upper left")
panel_label(ax, "a")
ax = fig.add_subplot(gs[1, 0])
vc = pd.Series(cp_unc["bootstrap_change_years"]).value_counts().sort_index()
ax.bar(vc.index, 100 * vc.values / vc.sum(), color=PAL["purple"], width=0.75)
ax.set(xlabel="Re-estimated change year", ylabel="Bootstrap replicates (%)")
ax.text(0.02, 0.97, f"Pettitt p < 0.05 in {cp_unc['pettitt_rejection_rate_under_pure_trend']:.0%}\nof pure-trend simulations",
        transform=ax.transAxes, va="top", fontsize=8, color=PAL["muted"])
panel_label(ax, "b")
ax = fig.add_subplot(gs[1, 1])
short = {"Broken (hinge) trend": "Hinge trend", "Step change (mean shift)": "Step", "Step change + common trend": "Step + trend",
         "Linear trend": "Linear trend", "Constant mean": "Constant"}
mm = cp_models.iloc[::-1]
ax.barh(mm.model.map(short), mm.BIC_weight, height=0.6,
        color=[PAL["purple"] if w == mm.BIC_weight.max() else PAL["purple_light"] for w in mm.BIC_weight])
for y, (w, d) in enumerate(zip(mm.BIC_weight, mm.delta_BIC)):
    ax.text(w + 0.02, y, f"ΔBIC {d:.1f}", va="center", fontsize=8, color=PAL["muted"])
ax.set(xlim=(0, 1.12), xlabel="BIC model weight")
panel_label(ax, "c")
save_figure(fig, "Fig04_shift_versus_trend", "Main",
            f"Abrupt shift versus gradual trend in annual rainfall. (a) Annual totals with segment means, the 1981–2025 Sen "
            f"trend and the best-BIC hinge model (break {hinge}); shading: 95% block-bootstrap range of the Pettitt change "
            f"year ({lo}–{hi}). (b) Bootstrap distribution of the change year ({N_BOOT} replicates). (c) BIC weights of five "
            f"competing models. The Pettitt test rejected in {cp_unc['pettitt_rejection_rate_under_pure_trend']:.0%} of "
            f"series simulated from a pure linear trend, so it cannot by itself distinguish a step from a trend.")
print("Saved Fig04.")

## 8–9. Dry pentads, wet pentads and rainfall concentration
Percentile dry pentads compare each pentad with the same calendar pentad in 1991–2020 (mid-rank percentile). About half of all calendar pentads (Oct–May) normally receive < 1 mm. In those pentads a "dry" pentad differs from a normal one only by fractions of a millimetre, which has no hydrological meaning, and such trace amounts are sensitive to changes in satellite or gauge inputs.

Section 8 therefore tests eight definitions: all pentads; only pentads with a normal ≥ 1 or ≥ 2 mm; and absolute < 1 or < 2 mm thresholds. It also tests the near-rainless component for homogeneity. Section 9 defines each concentration metric explicitly and separates the increase in rain from wet pentads (≥ 5 mm) into frequency and intensity.

In [ ]:
# =============================================================================
# 8. Dry pentads: definition sensitivity and the near-rainless-pentad problem
# =============================================================================
pct_reg = calendar_percentiles(R_reg, YEARS, NORMAL)
clim_pentad = R_reg[ref].mean(axis=0)[:, 0]
near_rainless = clim_pentad < NEAR_RAINLESS_MM
print(f"{near_rainless.sum()} of 72 calendar pentads have a 1991–2020 mean < {NEAR_RAINLESS_MM:g} mm "
      f"(months {sorted(int(m) for m in set(np.repeat(np.arange(1, 13), 6)[near_rainless]))}).")

dry_rows, dry_series = [], {}
for dname, d in DRY_DEFINITIONS.items():
    for sname, months in (("Annual", None), ("JJAS", SEASONS["JJAS"])):
        cnt, run = pentad_dryness(R_reg, YEARS, d, NORMAL, months=months, pct=pct_reg)
        for mname, x in (("count", cnt[:, 0]), ("longest run", run[:, 0])):
            dry_series[(dname, sname, mname)] = x
            r = trend_row(x, YEARS, f"{sname} dry-pentad {mname}", "pentads/yr", B=N_BOOT, seed=SEED)
            r.update(definition=dname, season=sname, metric=mname)
            dry_rows.append(r)
dry_sens = pd.DataFrame(dry_rows)
dry_sens["q_bh_within_definition"] = dry_sens.groupby("definition")["p_hr"].transform(bh_adjust)
display(dry_sens.pivot_table(index=["definition", "season"], columns="metric",
                             values=["sen_slope_per_year", "q_bh_within_definition"], sort=False).round(3))
save_table(dry_sens.round(5), "Table_S17_dry_pentad_definition_sensitivity", "Supplementary",
           "Trends in dry-pentad counts and longest runs under percentile definitions (calendar-pentad mid-rank percentile "
           "against 1991–2020; all pentads or only pentads with a 1991–2020 mean ≥1 or ≥2 mm) and absolute thresholds (<1, "
           "<2 mm). Ineligible pentads are not dry and break runs.")

dry_p20 = pct_reg[:, :, 0] <= 0.20
attrib = pd.DataFrame(dict(year=YEARS, all_pentads=dry_p20.sum(1), near_rainless=dry_p20[:, near_rainless].sum(1),
                           rain_bearing=dry_p20[:, ~near_rainless].sum(1)))
attrib_tr = trend_family([trend_row(attrib[c], YEARS, f"P20 dry pentads: {c}", "pentads/yr", B=N_BOOT)
                          for c in ("all_pentads", "near_rainless", "rain_bearing")], "P20 attribution")
hom = changepoint_battery(attrib.near_rainless.to_numpy(float), YEARS, B=N_PERM, seed=SEED).drop(columns="note")
print(f"Share of P20 dry pentads falling in near-rainless calendar pentads: {attrib.near_rainless.sum() / attrib.all_pentads.sum():.1%}")
display(attrib_tr[["series", "sen_slope_per_year", "sen_ci95_low", "sen_ci95_high", "p_hr"]].round(4))
display(hom.round(4))
save_table(attrib, "Table_S18_P20_dry_pentads_by_regime", "Supplementary",
           f"Annual P20 dry pentads split into near-rainless calendar pentads (1991–2020 mean < {NEAR_RAINLESS_MM:g} mm) and "
           "rain-bearing pentads.")
save_table(pd.concat([attrib_tr.round(5), hom.assign(series="near_rainless homogeneity").round(5)], ignore_index=True),
           "Table_S19_P20_attribution_and_homogeneity", "Supplementary",
           "Trends of the two P20 components and homogeneity tests (Pettitt, Buishand, SNHT; permutation p) of the "
           "near-rainless component.")

# =============================================================================
# 9. Rainfall concentration, frequency and intensity of wet pentads
# =============================================================================
CONC_DEFINITIONS = {
    "annual_total_mm": "sum of the 72 pentad totals (mm)",
    "n_wet_pentads_ge5": f"number of pentads with ≥{WET_PENTAD_MM:g} mm",
    "wet_pentad_intensity_mm": f"mean rainfall per pentad with ≥{WET_PENTAD_MM:g} mm (mm)",
    "rain_from_wet_pentads_mm": f"rain in pentads ≥{WET_PENTAD_MM:g} mm (mm)",
    "rain_from_sub5mm_pentads_mm": f"rain in pentads <{WET_PENTAD_MM:g} mm (mm)",
    "top10_share_pct": "100 × sum of the 10 largest pentads / annual total",
    "max_pentad_mm": "largest pentad total (mm)",
    "gini_pentad": "Gini coefficient of the 72 pentad totals",
    "pci_monthly": "Precipitation Concentration Index (Oliver 1980), 100·Σm²/(Σm)²",
}
conc = {k: v[:, 0] for k, v in concentration_metrics(R_reg, WET_PENTAD_MM).items()}
conc_tr = trend_family([trend_row(conc[k], YEARS, k, "per yr", B=N_BOOT, seed=SEED) for k in CONC_DEFINITIONS],
                       "Rainfall concentration")
conc_tr.insert(2, "definition", conc_tr.series.map(CONC_DEFINITIONS))
display(conc_tr[["series", "sen_slope_per_year", "sen_ci95_low", "sen_ci95_high", "p_hr", "q_bh_within_family"]].round(3))
decomp = []
ok = conc["n_wet_pentads_ge5"] > 0
for k in ("rain_from_wet_pentads_mm", "n_wet_pentads_ge5", "wet_pentad_intensity_mm"):
    bci = mbb_sen_ci(np.log(conc[k][ok]), B=N_BOOT, seed=SEED)
    decomp.append(dict(component=k, pct_per_decade=100 * (np.exp(10 * bci["sen_slope"]) - 1),
                       ci95_low=100 * (np.exp(10 * bci["ci_low"]) - 1), ci95_high=100 * (np.exp(10 * bci["ci_high"]) - 1)))
decomp = pd.DataFrame(decomp)
display(decomp.round(2))
save_table(pd.DataFrame(dict(year=YEARS, **conc)).round(4), "Table_S20_concentration_metrics", "Supplementary",
           "Annual rainfall-concentration metrics; definitions in Table S21.")
save_table(conc_tr.round(5), "Table_S21_concentration_trends", "Supplementary",
           "Trends in concentration metrics (definitions given), Hamed–Rao MK with block-bootstrap Sen CIs, BH q within family.")
save_table(decomp.round(3), "Table_S22_frequency_intensity", "Supplementary",
           f"Relative Sen trends (% per decade, 95% block-bootstrap CI) of rain in wet pentads (≥{WET_PENTAD_MM:g} mm), their "
           "number and their mean intensity, estimated on log scale. Log totals are additive, but separately estimated Sen "
           "slopes need not sum exactly.")

# ---- Figure: frequency vs intensity -----------------------------------------------------------
fig = new_figure("double", 80)
gs = fig.add_gridspec(1, 2, width_ratios=[1.75, 1])
ax = fig.add_subplot(gs[0])
ax.bar(YEARS, conc["rain_from_sub5mm_pentads_mm"], color=PAL["purple_light"], width=0.8, label=f"Pentads <{WET_PENTAD_MM:g} mm")
ax.bar(YEARS, conc["rain_from_wet_pentads_mm"], bottom=conc["rain_from_sub5mm_pentads_mm"], color=PAL["purple"], width=0.8,
       label=f"Pentads ≥{WET_PENTAD_MM:g} mm")
ax.set(ylabel="Annual rainfall (mm)", xlim=(YEARS[0] - 1, YEARS[-1] + 1))
ax.legend(loc="upper left")
panel_label(ax, "a")
ax = fig.add_subplot(gs[1])
lab = {"rain_from_wet_pentads_mm": "Wet-pentad rain", "n_wet_pentads_ge5": "Frequency", "wet_pentad_intensity_mm": "Intensity"}
y = np.arange(len(decomp))
ax.errorbar(decomp.pct_per_decade, y, xerr=[decomp.pct_per_decade - decomp.ci95_low, decomp.ci95_high - decomp.pct_per_decade],
            fmt="o", color=PAL["purple_dark"], ms=3.5, elinewidth=0.9, capsize=0)
ax.set_yticks(y, [lab[c] for c in decomp.component])
ax.invert_yaxis()
ax.axvline(0, color=PAL["muted"], lw=0.5)
ax.set_xlabel("Sen trend (% per decade, 95% CI)")
panel_label(ax, "b")
_wet_slope = conc_tr.set_index("series").loc["rain_from_wet_pentads_mm", "sen_slope_per_year"]
save_figure(fig, "Fig05_wet_pentad_frequency_intensity", "Main",
            f"Source of the rainfall increase. (a) Annual rainfall split into pentads <{WET_PENTAD_MM:g} mm and ≥{WET_PENTAD_MM:g} mm "
            f"(CHIRPS v2). (b) Relative Sen trends with 95% block-bootstrap CIs of rain in wet pentads, their number "
            f"(frequency) and mean amount (intensity). Rain in wet pentads increased by {_wet_slope:.2f} mm yr⁻¹ of the "
            f"{rain_trends.set_index('series').loc['Annual rainfall', 'sen_slope_per_year']:.2f} mm yr⁻¹ annual trend.")
print("Saved Fig05.")

## 10–12. Spatial analysis on the native CHIRPS grid
Grid cells ≥ 50 % inside the boundary are analysed with the regional definitions. Pixel trends are corrected for multiplicity with BH-FDR; q ≤ 0.10 is used for field significance (Wilks 2016).

Local Moran's I (queen contiguity, conditional permutation, BH-FDR) identifies clusters of relatively strong or weak trends. Because CHIRPS cells share inputs, a large global Moran's I is expected. The cluster *locations* carry the information, and their boundaries are approximate.

In [ ]:
# =============================================================================
# 10. Native CHIRPS grid, boundary-coverage weights and pentad cubes (v2 and v3)
# -----------------------------------------------------------------------------
# The native 0.05° pentad record for the Cholistan window (~45 yr × 72 pentads ×
# ~2,800 cells) is read from the local cache; if absent and Earth Engine is allowed,
# it is downloaded with one 72-band ee.data.computePixels request per year.
# Coverage fractions come from the boundary shapefile (20×20 sub-cell sampling) and
# area weights = coverage × spherical cell area.
# =============================================================================
import struct
from matplotlib.path import Path as MplPath

def read_polygon_shapefile(shp_path):
    """Minimal ESRI Polygon (type 5) reader -> list of rings [(N,2) lon/lat arrays]
    with ring orientation, so no GDAL/geopandas is required."""
    b = Path(shp_path).read_bytes()
    if struct.unpack("<i", b[32:36])[0] not in (5, 15, 25):
        raise ValueError("Boundary must be a polygon shapefile")
    pos, rings = 100, []
    while pos < len(b):
        _, clen = struct.unpack(">2i", b[pos:pos + 8]); pos += 8
        rec = b[pos:pos + 2 * clen]; pos += 2 * clen
        if struct.unpack("<i", rec[:4])[0] == 0:
            continue
        nparts, npts = struct.unpack("<2i", rec[36:44])
        parts = list(struct.unpack(f"<{nparts}i", rec[44:44 + 4 * nparts])) + [npts]
        xy = np.frombuffer(rec[44 + 4 * nparts:44 + 4 * nparts + 16 * npts], dtype="<f8").reshape(-1, 2)
        rings += [xy[parts[i]:parts[i + 1]] for i in range(nparts)]
    return rings


def ring_is_hole(ring):          # ESRI: outer rings clockwise, holes counter-clockwise
    x, y = ring[:, 0], ring[:, 1]
    return 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1]) > 0


def coverage_fraction(rings, lon_edges, lat_edges, sub=20):
    """Fraction of each grid cell inside the polygon (sub×sub point sampling;
    resolution 0.25 % of a cell). Holes are subtracted."""
    H, W = len(lat_edges) - 1, len(lon_edges) - 1
    fx = (np.arange(sub) + 0.5) / sub
    lon_s = (lon_edges[:-1, None] + np.diff(lon_edges)[:, None] * fx[None, :]).ravel()   # W*sub
    lat_s = (lat_edges[:-1, None] + np.diff(lat_edges)[:, None] * fx[None, :]).ravel()   # H*sub
    LON, LAT = np.meshgrid(lon_s, lat_s)
    pts = np.c_[LON.ravel(), LAT.ravel()]
    inside = np.zeros(len(pts), bool)
    for ring in rings:
        hit = MplPath(ring).contains_points(pts)
        inside = (inside & ~hit) if ring_is_hole(ring) else (inside | hit)
    return inside.reshape(H, sub, W, sub).mean(axis=(1, 3))


def cell_area_m2(lat_edges, lon_step_deg, H, W):
    R = 6371008.8
    a = R ** 2 * np.deg2rad(abs(lon_step_deg)) * np.abs(np.sin(np.deg2rad(lat_edges[:-1])) - np.sin(np.deg2rad(lat_edges[1:])))
    return np.repeat(a[:, None], W, axis=1)


def cache_complete(folder, tag):
    return (folder / f"{tag}_cube_{START_YEAR}_{END_YEAR}.npz").is_file() or all(
        (folder / f"{tag}_{int(y)}.npy").is_file() for y in YEARS)


CACHE = next((c for c in CACHE_CANDIDATES if (c / "chirps_native_grid.json").is_file()
              and cache_complete(c, "chirps_v2")), DIRS["cache"])
print("Spatial cache:", CACHE)

rings = read_polygon_shapefile(BOUNDARY_SHP)
_xy = np.vstack(rings)
if (CACHE / "chirps_native_grid.json").is_file():
    grid_meta = json.loads((CACHE / "chirps_native_grid.json").read_text())
else:
    ee = ee_init()
    proj = ee.Image(ee.ImageCollection(CHIRPS_V2_PENTAD).first()).select("precipitation").projection().getInfo()
    sx, _, x0, _, sy, y0 = proj["transform"]
    (minx, miny), (maxx, maxy) = _xy.min(axis=0), _xy.max(axis=0)
    c0, c1 = int(np.floor((minx - x0) / sx)) - 1, int(np.ceil((maxx - x0) / sx)) + 1
    r0, r1 = int(np.floor((maxy - y0) / sy)) - 1, int(np.ceil((miny - y0) / sy)) + 1
    grid_meta = dict(crs=proj.get("crs", "EPSG:4326"), sx=sx, sy=sy, x0=x0 + c0 * sx, y0=y0 + r0 * sy,
                     width=c1 - c0, height=r1 - r0, source_transform=proj["transform"])
    (CACHE / "chirps_native_grid.json").write_text(json.dumps(grid_meta, indent=2))

Hg, Wg = grid_meta["height"], grid_meta["width"]
lon_edges = grid_meta["x0"] + np.arange(Wg + 1) * grid_meta["sx"]
lat_edges = grid_meta["y0"] + np.arange(Hg + 1) * grid_meta["sy"]
lon_c, lat_c = 0.5 * (lon_edges[:-1] + lon_edges[1:]), 0.5 * (lat_edges[:-1] + lat_edges[1:])
frac = coverage_fraction(rings, lon_edges, lat_edges)
weights = frac * cell_area_m2(lat_edges, grid_meta["sx"], Hg, Wg)
analysis_mask = frac >= ANALYSIS_MIN_FRACTION


def fetch_pentad_year(collection_id, year, tag):
    f = CACHE / f"{tag}_{year}.npy"
    if f.exists():
        return np.load(f, allow_pickle=False)
    ee = ee_init()
    col = (ee.ImageCollection(collection_id).filterDate(f"{year}-01-01", f"{year + 1}-01-01")
           .select("precipitation").sort("system:time_start"))
    labels = [pd.Timestamp(t, unit="ms").strftime("P%Y%m%d") for t in col.aggregate_array("system:time_start").getInfo()]
    if labels != [f"P{year}{m:02d}{d:02d}" for m in range(1, 13) for d in (1, 6, 11, 16, 21, 26)]:
        raise ValueError(f"{tag} {year}: unexpected pentad calendar ({len(labels)} images)")
    img = col.toBands().rename(labels).unmask(-9999).toFloat()
    grid = {"dimensions": {"width": Wg, "height": Hg}, "crsCode": grid_meta["crs"],
            "affineTransform": {"scaleX": grid_meta["sx"], "shearX": 0, "translateX": grid_meta["x0"],
                                "shearY": 0, "scaleY": grid_meta["sy"], "translateY": grid_meta["y0"]}}
    arr = ee_retry(lambda: ee.data.computePixels({"expression": img, "fileFormat": "NUMPY_NDARRAY", "grid": grid}),
                   f"{tag} {year}")
    cube = np.stack([np.asarray(arr[l], dtype="float32") for l in labels])
    cube[cube <= -9000] = np.nan
    np.save(f, cube)
    return cube


def load_cube(collection_id, tag, required=True):
    final = CACHE / f"{tag}_cube_{START_YEAR}_{END_YEAR}.npz"
    if final.exists():
        with np.load(final, allow_pickle=False) as z:
            cube = z["cube"]
            if cube.shape != (len(YEARS), 72, Hg, Wg) or not np.array_equal(z["years"], YEARS):
                raise ValueError(f"Cached cube does not match the grid/calendar: {final}")
            return cube
    if not cache_complete(CACHE, tag) and not ALLOW_EARTH_ENGINE:
        if required:
            raise FileNotFoundError(f"{tag} cube missing in {CACHE}; restore it or set ALLOW_EARTH_ENGINE = True.")
        print(f"[skipped] {tag}: not cached")
        return None
    cube = np.stack([fetch_pentad_year(collection_id, int(y), tag) for y in YEARS])
    np.savez_compressed(final, cube=cube, years=YEARS, lon=lon_c, lat=lat_c)
    return cube


cube_v2 = load_cube(CHIRPS_V2_PENTAD, "chirps_v2")
cube_v3 = load_cube(CHIRPS_V3_PENTAD, "chirps_v3", required=False)
inside = frac > 0
for tag, cube in (("CHIRPS v2", cube_v2), ("CHIRPS v3", cube_v3)):
    if cube is not None and (not np.isfinite(cube[:, :, inside]).all() or (cube[:, :, inside] < 0).any()):
        raise ValueError(f"{tag}: invalid rainfall inside the boundary")


def regional_series(cube, w):
    m = w > 0
    return np.stack([(yr[:, m] * w[m]).sum(axis=-1) / w[m].sum() for yr in cube])        # (Y, 72)


reg_v2 = regional_series(cube_v2, weights)
_d = reg_v2.ravel() - R_reg[:, :, 0].ravel()
grid_qc = pd.DataFrame([dict(grid=f"{Hg}×{Wg} cells at {grid_meta['sx']}°", cells_touching=int(inside.sum()),
                             analysis_cells=int(analysis_mask.sum()), area_km2=weights.sum() / 1e6,
                             r_vs_archived=np.corrcoef(reg_v2.ravel(), R_reg[:, :, 0].ravel())[0, 1],
                             max_abs_diff_mm=np.abs(_d).max(), mean_diff_mm=_d.mean())])
display(grid_qc.round(4))
save_table(grid_qc.round(4), "Table_S23_native_grid_quality_control", "Supplementary",
           "Native CHIRPS grid: coverage, analysed cells and agreement of the grid-derived area-weighted pentad series "
           "with the archived Earth Engine regional series.")

In [ ]:
# =============================================================================
# 11. Pixel-wise trends (CHIRPS v2 and v3), FDR, and spatial clustering (LISA)
# -----------------------------------------------------------------------------
# Every metric uses the regional definitions above, evaluated per grid cell (SPI is
# fitted per cell and ending month). BH-FDR is applied over cells; q ≤ 0.10 is the
# field-significance criterion of Wilks (2016) and q ≤ 0.05 the stricter one.
# Sen slopes of zero-inflated drought-month counts are degenerate, so SPI severity
# is mapped as the annual minimum SPI. Local Moran's I (queen contiguity, conditional
# permutation, BH-FDR) identifies clusters of high or low trends. CHIRPS cells are
# correlated by construction (blended climatology and sparse gauges): a large global
# Moran's I is expected, and cluster boundaries are approximate.
# =============================================================================
from scipy import sparse

pix_r, pix_c = np.where(analysis_mask)
P = len(pix_r)
pix_lon, pix_lat, pix_w = lon_c[pix_c], lat_c[pix_r], weights[pix_r, pix_c]
MAP_METRICS = {   # name: (role, drier_when_positive)
    "Annual rainfall (mm)": ("Main", False), "JJAS rainfall (mm)": ("Main", False),
    "Wet pentads ≥5 mm (count)": ("Main", False), "SPI-12 annual minimum (index)": ("Main", False),
    "Top-10 pentad share (%)": ("Supplementary", False), "JJAS longest dry run <2 mm (pentads)": ("Supplementary", True),
    "P20 dry pentads, all pentads (count)": ("Supplementary", True), "P20 dry pentads, normal ≥1 mm (count)": ("Supplementary", True),
    "SPI-3 annual minimum (index)": ("Supplementary", False),
}


def pixel_metrics(cube):
    Rp = cube[:, :, pix_r, pix_c].astype(float)
    mon = Rp.reshape(len(YEARS), 12, 6, P).sum(axis=2)
    cm = concentration_metrics(Rp, WET_PENTAD_MM)
    pct = calendar_percentiles(Rp, YEARS, NORMAL)
    out = {"Annual rainfall (mm)": (YEARS, Rp.sum(axis=1)),
           "JJAS rainfall (mm)": (YEARS, mon[:, 5:9].sum(axis=1)),
           "Wet pentads ≥5 mm (count)": (YEARS, cm["n_wet_pentads_ge5"]),
           "Top-10 pentad share (%)": (YEARS, cm["top10_share_pct"]),
           "JJAS longest dry run <2 mm (pentads)": (YEARS, pentad_dryness(Rp, YEARS, DRY_DEFINITIONS["Absolute <2 mm"], NORMAL,
                                                                          months=SEASONS["JJAS"])[1]),
           "P20 dry pentads, all pentads (count)": (YEARS, pentad_dryness(Rp, YEARS, DRY_DEFINITIONS["P20 (all pentads)"], NORMAL, pct=pct)[0]),
           "P20 dry pentads, normal ≥1 mm (count)": (YEARS, pentad_dryness(Rp, YEARS, DRY_DEFINITIONS["P20 (normal ≥1 mm)"], NORMAL, pct=pct)[0])}
    flat = mon.reshape(len(YEARS) * 12, P)
    for k in (3, 12):
        s, _ = spi(flat, np.tile(np.arange(1, 13), len(YEARS)), np.repeat(YEARS, 12), k, calibration=SPI_PRIMARY_CALIBRATION)
        out[f"SPI-{k} annual minimum (index)"] = (YEARS[1:], np.nanmin(s.reshape(len(YEARS), 12, P), axis=1)[1:])
    return out


def pixel_trends(metrics):
    rows, maps = [], {}
    w = pix_w / pix_w.sum()
    for name, (yrs, X) in metrics.items():
        res = mk_hamed_rao(X)
        q = bh_adjust(res["p_hr"])
        sd = 10 * res["sen_slope"]
        maps[name] = dict(slope_per_decade=sd, p=res["p_hr"], q=q)
        rows.append(dict(metric=name, years=f"{yrs[0]}–{yrs[-1]}", cells=P, mean_slope_per_decade=np.nansum(w * sd),
                         slope_p10=np.nanpercentile(sd, 10), slope_p90=np.nanpercentile(sd, 90),
                         pct_area_positive=100 * w[sd > 0].sum(), pct_area_negative=100 * w[sd < 0].sum(),
                         pct_area_sig_increase_q10=100 * w[(q <= .10) & (sd > 0)].sum(),
                         pct_area_sig_decrease_q10=100 * w[(q <= .10) & (sd < 0)].sum(),
                         pct_area_sig_increase_q05=100 * w[(q <= .05) & (sd > 0)].sum(),
                         pct_area_sig_decrease_q05=100 * w[(q <= .05) & (sd < 0)].sum()))
    return pd.DataFrame(rows), maps


metrics_v2 = pixel_metrics(cube_v2)
trend_summary_v2, tmaps_v2 = pixel_trends(metrics_v2)
display(trend_summary_v2.round(2))
save_table(trend_summary_v2.round(3), "Table_S24_pixel_trend_summary_CHIRPSv2", "Supplementary",
           f"Pixel-wise Hamed–Rao MK and Sen slopes (per decade) over {P} CHIRPS v2 cells (≥{ANALYSIS_MIN_FRACTION:.0%} inside the "
           "boundary): area-weighted mean slope, 10th–90th percentile of slopes, and % of area with significant increase/"
           "decrease after BH-FDR (q ≤ 0.10, field significance; q ≤ 0.05).")
tmaps_v3 = None
if cube_v3 is not None:
    metrics_v3 = pixel_metrics(cube_v3)
    trend_summary_v3, tmaps_v3 = pixel_trends(metrics_v3)
    agree = []
    for name in tmaps_v2:
        a, b = tmaps_v2[name]["slope_per_decade"], tmaps_v3[name]["slope_per_decade"]
        ok = np.isfinite(a) & np.isfinite(b)
        agree.append(dict(metric=name, v2_mean_slope=np.nanmean(a), v3_mean_slope=np.nanmean(b),
                          pct_cells_same_sign=100 * np.mean(np.sign(a[ok]) == np.sign(b[ok])),
                          spatial_r=np.corrcoef(a[ok], b[ok])[0, 1] if a[ok].std() > 0 and b[ok].std() > 0 else np.nan,
                          v2_pct_area_sig_q10=trend_summary_v2.set_index("metric").loc[name, ["pct_area_sig_increase_q10", "pct_area_sig_decrease_q10"]].sum(),
                          v3_pct_area_sig_q10=trend_summary_v3.set_index("metric").loc[name, ["pct_area_sig_increase_q10", "pct_area_sig_decrease_q10"]].sum()))
    v2v3_pixels = pd.DataFrame(agree)
    display(v2v3_pixels.round(2))
    save_table(trend_summary_v3.round(3), "Table_S25_pixel_trend_summary_CHIRPSv3", "Supplementary",
               "As Table S24 for CHIRPS v3 on the same grid.")
    save_table(v2v3_pixels.round(3), "Table_S26_pixel_trends_v2_v3_agreement", "Supplementary",
               "Agreement of pixel-wise trends between CHIRPS v2 and v3: mean slopes, % of cells with the same sign, spatial "
               "correlation of slopes, and % of area significant (q ≤ 0.10).")

def grid_weights(rows, cols, queen=True):
    """Binary contiguity among analysis cells -> (binary csr, row-standardised csr)."""
    idx = {(r, c): i for i, (r, c) in enumerate(zip(rows, cols))}
    offs = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if queen else [])
    I, J = [], []
    for i, (r, c) in enumerate(zip(rows, cols)):
        for dr, dc in offs:
            j = idx.get((r + dr, c + dc))
            if j is not None:
                I.append(i); J.append(j)
    n = len(rows)
    A = sparse.csr_matrix((np.ones(len(I)), (I, J)), shape=(n, n))
    deg = np.asarray(A.sum(axis=1)).ravel()
    Wr = sparse.diags(np.where(deg > 0, 1 / np.maximum(deg, 1), 0)) @ A
    return A, Wr.tocsr(), deg


def moran_global(y, Wr, perms=9999, seed=SEED):
    z = y - y.mean()
    n = len(z)
    s0 = Wr.sum()
    I = n / s0 * (z @ (Wr @ z)) / (z @ z)
    rng = np.random.default_rng(seed)
    sims = np.empty(perms)
    for s in range(0, perms, 500):
        Z = np.stack([rng.permutation(z) for _ in range(min(500, perms - s))], axis=1)
        sims[s:s + Z.shape[1]] = n / s0 * np.einsum("ij,ij->j", Z, Wr @ Z) / (z @ z)
    return dict(I=I, E_I=-1 / (n - 1), z_sim=(I - sims.mean()) / sims.std(),
                p_sim=(1 + np.sum(sims >= I)) / (perms + 1))


def lisa(y, Wr, deg, perms=9999, seed=SEED):
    """Local Moran's I with conditional permutation (Anselin 1995), folded
    pseudo p-values as in PySAL/esda (one random draw of neighbour sets reused
    across locations, exactly as esda's 'crand')."""
    z = (y - y.mean()) / y.std()
    n = len(z)
    lag = Wr @ z
    Is = z * lag
    rng = np.random.default_rng(seed)
    kmax = int(deg.max())
    rids = np.stack([rng.permutation(n - 1)[:kmax] for _ in range(perms)])
    p = np.full(n, np.nan)
    for i in range(n):
        k = int(deg[i])
        if k == 0:
            continue
        ids = rids[:, :k].copy()
        ids[ids >= i] += 1
        sim = z[i] * z[ids].mean(axis=1)
        larger = np.sum(sim >= Is[i])
        larger = min(larger, perms - larger)
        p[i] = (larger + 1) / (perms + 1)
    quad = np.select([(z > 0) & (lag > 0), (z < 0) & (lag > 0), (z < 0) & (lag < 0), (z > 0) & (lag < 0)],
                     ["High–High", "Low–High", "Low–Low", "High–Low"], "None")
    return dict(Ii=Is, p_sim=p, q_bh=bh_adjust(p), quadrant=quad)



A_queen, W_queen, deg_queen = grid_weights(pix_r, pix_c, queen=True)
A_rook, W_rook, deg_rook = grid_weights(pix_r, pix_c, queen=False)
LISA_METRICS = [m for m, (role, _) in MAP_METRICS.items() if role == "Main"]
moran_rows, lisa_maps = [], {}
for name in LISA_METRICS:
    y = tmaps_v2[name]["slope_per_decade"]
    if not np.isfinite(y).all() or np.std(y) < 1e-12:
        print(f"{name}: slope field undefined or constant — clustering skipped")
        continue
    for label, (W_, d_) in {"queen": (W_queen, deg_queen), "rook": (W_rook, deg_rook)}.items():
        g = moran_global(y, W_, perms=LISA_PERMUTATIONS if label == "queen" else 999)
        L = lisa(y, W_, d_, perms=LISA_PERMUTATIONS)
        sig = (L["q_bh"] <= 0.05) & (L["quadrant"] != "None")
        row = dict(metric=name, contiguity=label, moran_I=g["I"], p_sim=g["p_sim"], lisa_cells_q05=int(sig.sum()))
        for qd in ("High–High", "Low–Low", "High–Low", "Low–High"):
            row[f"{qd}_area_pct"] = 100 * pix_w[sig & (L["quadrant"] == qd)].sum() / pix_w.sum()
        moran_rows.append(row)
        if label == "queen":
            lisa_maps[name] = L
moran_table = pd.DataFrame(moran_rows)
display(moran_table.round(3))
save_table(moran_table.round(4), "Table_S27_moran_lisa_summary", "Supplementary",
           f"Global Moran's I of Sen-slope fields (permutation p) and area of significant Local Moran's I clusters "
           f"({LISA_PERMUTATIONS} conditional permutations, BH-FDR q ≤ 0.05); queen (primary) and rook (sensitivity) contiguity.")

pix_table = pd.DataFrame(dict(lon=pix_lon, lat=pix_lat, coverage_fraction=frac[pix_r, pix_c], weight_m2=pix_w))
for name, m in tmaps_v2.items():
    key = name.split(" (")[0]
    pix_table[f"{key} | slope per decade"] = m["slope_per_decade"]
    pix_table[f"{key} | q_BH"] = m["q"]
    if name in lisa_maps:
        pix_table[f"{key} | LISA cluster (q≤0.05)"] = np.where(lisa_maps[name]["q_bh"] <= 0.05, lisa_maps[name]["quadrant"], "n.s.")
save_table(pix_table.round(5), "Table_S28_pixel_values_CHIRPSv2", "Supplementary",
           "Cell-level values: centre coordinates, coverage, area weight, Sen slope per decade, BH q and LISA class for each metric.")

In [ ]:
# =============================================================================
# 12. Trend and cluster maps
# =============================================================================
from matplotlib.colors import TwoSlopeNorm, ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

LISA_COLORS = {"Not significant": "#E4E1E4", "High–High": "#B2182B", "Low–Low": "#2166AC",
               "High–Low": "#EF8A62", "Low–High": "#67A9CF"}
UNITS = {"(mm)": "mm decade$^{-1}$", "(count)": "pentads decade$^{-1}$", "(pentads)": "pentads decade$^{-1}$",
         "(%)": "%-points decade$^{-1}$", "(index)": "SPI decade$^{-1}$"}
LON_LIM = (lon_edges.min(), lon_edges.max())
LAT_LIM = (lat_edges.min(), lat_edges.max())


def to_grid(v):
    g = np.full((Hg, Wg), np.nan)
    g[pix_r, pix_c] = v
    return g


def style_map(ax, labels=True):
    ax.set_aspect(1 / np.cos(np.deg2rad(lat_c.mean())))
    ax.set_xlim(LON_LIM); ax.set_ylim(LAT_LIM)
    for ring in rings:
        if not ring_is_hole(ring):
            ax.plot(ring[:, 0], ring[:, 1], color=PAL["ink"], lw=0.6)
    ax.set_xticks(np.arange(70, 73.1, 1)); ax.set_yticks(np.arange(28, 29.6, 0.5))
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0f}°E")
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.1f}°N")
    for s in ("top", "right"):
        ax.spines[s].set_visible(True)
    if not labels:
        ax.set_yticklabels([])


def trend_panel(fig, ax, name, maps, title=None, lim=None):
    v = maps[name]["slope_per_decade"]
    drier = MAP_METRICS[name][1]
    lim = lim or (np.nanpercentile(np.abs(v), 98) or 1.0)
    im = ax.pcolormesh(lon_edges, lat_edges, to_grid(v), cmap="BrBG_r" if drier else "BrBG",
                       norm=TwoSlopeNorm(0, -lim, lim), shading="flat", rasterized=True)
    sig = maps[name]["q"] <= 0.10
    ax.scatter(pix_lon[sig], pix_lat[sig], s=0.25, c=PAL["ink"], lw=0, alpha=0.8)
    style_map(ax)
    cb = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.015, aspect=22)
    cb.set_label(next(u for k, u in UNITS.items() if name.endswith(k)), fontsize=8)
    cb.ax.tick_params(labelsize=8)
    ax.set_title(title or name.split(" (")[0])
    return im


def trend_map_figure(names, maps, fname, role, caption, titles=None):
    nrow = int(np.ceil(len(names) / 2))
    fig = new_figure("double", 68 * nrow + 10)
    axes = fig.subplots(nrow, 2, squeeze=False).ravel()
    maps = maps if isinstance(maps, list) else [maps] * len(names)
    # identical colour limits for panels that show the same metric (e.g. v2 vs v3)
    lims = {n: max(np.nanpercentile(np.abs(m[n]["slope_per_decade"]), 98) for m, nn in zip(maps, names) if nn == n) or 1.0
            for n in set(names)}
    for i, (ax, name) in enumerate(zip(axes, names)):
        trend_panel(fig, ax, name, maps[i], title=titles[i] if titles else None, lim=lims[name])
        panel_label(ax, "abcdefgh"[i], x=-0.03)
    for ax in axes[len(names):]:
        ax.set_axis_off()
    return save_figure(fig, fname, role, caption)


def _area_sig(name, summary=trend_summary_v2):
    r = summary.set_index("metric").loc[name]
    return r.pct_area_sig_increase_q10 + r.pct_area_sig_decrease_q10


main_names = [m for m, (role, _) in MAP_METRICS.items() if role == "Main"]
trend_map_figure(main_names, tmaps_v2, "Fig06_pixel_trend_maps", "Main",
                 f"Pixel-wise Sen slopes (per decade), CHIRPS v2, 1981–2025, for (a) annual rainfall, (b) JJAS rainfall, "
                 f"(c) number of wet pentads (≥{WET_PENTAD_MM:g} mm) and (d) annual minimum SPI-12 (1982–2025). Dots: cells "
                 f"significant after BH-FDR at q ≤ 0.10 (Hamed–Rao MK); area significant: "
                 + ", ".join(f"{_area_sig(n):.0f}%" for n in main_names) + ". Green: wetter/less severe.")

fig = new_figure("double", 142)
axes = fig.subplots(2, 2).ravel()
for i, (ax, name) in enumerate(zip(axes, [n for n in LISA_METRICS if n in lisa_maps])):
    L = lisa_maps[name]
    cls = np.where((L["q_bh"] <= 0.05) & (L["quadrant"] != "None"), L["quadrant"], "Not significant")
    codes = list(LISA_COLORS)
    ax.pcolormesh(lon_edges, lat_edges, to_grid(np.array([codes.index(c) for c in cls], float)),
                  cmap=ListedColormap(list(LISA_COLORS.values())), norm=BoundaryNorm(np.arange(-0.5, 5.5), 5),
                  shading="flat", rasterized=True)
    style_map(ax)
    ax.set_title(name.split(" (")[0])
    panel_label(ax, "abcd"[i], x=-0.03)
fig.legend(handles=[Patch(facecolor=c, label=l) for l, c in LISA_COLORS.items()], loc="outside lower center", ncol=5)
save_figure(fig, "Fig07_LISA_trend_clusters", "Main",
            f"Local Moran's I clusters of the trend fields in Fig. 6 (queen contiguity, {LISA_PERMUTATIONS} conditional "
            "permutations, BH-FDR q ≤ 0.05). High–High (Low–Low): cells whose slope and neighbourhood slope both exceed "
            "(fall below) the Cholistan mean slope; they describe relative strength of change, not its sign.")

supp_names = [m for m, (role, _) in MAP_METRICS.items() if role == "Supplementary"]
trend_map_figure(supp_names[:4], tmaps_v2, "FigS2_pixel_trend_maps_supplementary", "Supplementary",
                 "Pixel-wise Sen slopes (per decade), CHIRPS v2, for (a) top-10 pentad share, (b) longest JJAS run of "
                 "pentads < 2 mm, (c) P20 dry pentads counted over all calendar pentads and (d) P20 dry pentads restricted to "
                 "pentads with a 1991–2020 mean ≥ 1 mm. Dots: BH-FDR q ≤ 0.10. Brown: drier tendency.")
if tmaps_v3 is not None:
    names = ["Annual rainfall (mm)", "Annual rainfall (mm)", "P20 dry pentads, all pentads (count)", "P20 dry pentads, all pentads (count)"]
    trend_map_figure(names, [tmaps_v2, tmaps_v3, tmaps_v2, tmaps_v3], "FigS3_CHIRPS_v2_v3_trend_maps", "Supplementary",
                     "CHIRPS v2 (a, c) versus CHIRPS v3 (b, d) Sen slopes per decade on the same 0.05° grid: annual rainfall "
                     "(a, b) and P20 dry pentads counted over all calendar pentads (c, d). Dots: BH-FDR q ≤ 0.10.",
                     titles=["Annual rainfall — CHIRPS v2", "Annual rainfall — CHIRPS v3",
                             "P20 dry pentads (all) — CHIRPS v2", "P20 dry pentads (all) — CHIRPS v3"])
print("Saved map figures.")

## 13. Robustness to dataset version and to independent products
No rain-gauge record was available. Two checks are used instead:
* **Version replication.** Key metrics are re-estimated with CHIRPS v3 (Table 3).
* **Product agreement.** CHIRPS v2 is compared with PERSIANN-CDR, ERA5-Land and IMERG V07 (Table 4, Fig. 8), which have different error structures.

Agreement shows that a signal is not specific to one product; it is not a validation against observations.

In [ ]:
# =============================================================================
# 13. Robustness: CHIRPS v2 vs v3 (regional) and agreement with independent products
# -----------------------------------------------------------------------------
# No rain-gauge record was available. Agreement among products with different
# error structures (CHIRPS v3; PERSIANN-CDR: IR + GPCP adjustment; ERA5-Land:
# reanalysis; IMERG V07 Final: microwave + IR, GPCC-adjusted) indicates whether a
# signal is product-specific. It is not accuracy against observations.
# =============================================================================
import calendar


def regional_metrics(Rp):
    """Key annual metrics from a regional (Y, 72, 1) pentad array."""
    mon = Rp[:, :, 0].reshape(len(YEARS), 12, 6).sum(axis=2)
    cm = {k: v[:, 0] for k, v in concentration_metrics(Rp, WET_PENTAD_MM).items()}
    pc = calendar_percentiles(Rp, YEARS, NORMAL)
    nr = Rp[ref].mean(axis=0)[:, 0] < NEAR_RAINLESS_MM
    s12, _ = spi(mon.ravel(), np.tile(np.arange(1, 13), len(YEARS)), np.repeat(YEARS, 12), 12)
    out = {"Annual rainfall (mm)": cm["annual_total_mm"], "JJAS rainfall (mm)": mon[:, 5:9].sum(1),
           "Wet pentads ≥5 mm (count)": cm["n_wet_pentads_ge5"], "Wet-pentad intensity (mm)": cm["wet_pentad_intensity_mm"],
           "Top-10 pentad share (%)": cm["top10_share_pct"], "Gini of pentad totals": cm["gini_pentad"],
           "P20 dry pentads, all pentads (count)": (pc[:, :, 0] <= .2).sum(1).astype(float),
           "P20 dry pentads, near-rainless pentads (count)": (pc[:, nr, 0] <= .2).sum(1).astype(float),
           "P20 dry pentads, normal ≥1 mm (count)": pentad_dryness(Rp, YEARS, DRY_DEFINITIONS["P20 (normal ≥1 mm)"], NORMAL, pct=pc)[0][:, 0],
           "JJAS longest dry run <2 mm (pentads)": pentad_dryness(Rp, YEARS, DRY_DEFINITIONS["Absolute <2 mm"], NORMAL,
                                                                  months=SEASONS["JJAS"])[1][:, 0],
           "SPI-12 annual minimum": np.r_[np.nan, np.nanmin(s12.reshape(len(YEARS), 12), axis=1)[1:]]}
    return out


robust_rows = []
reg_metrics = {"CHIRPS v2": regional_metrics(R_reg)}
if cube_v3 is not None:
    reg_v3 = regional_series(cube_v3, weights)
    reg_metrics["CHIRPS v3"] = regional_metrics(reg_v3[:, :, None])
for metric in reg_metrics["CHIRPS v2"]:
    row = dict(metric=metric)
    for prod, mets in reg_metrics.items():
        x = mets[metric]
        tr = trend_row(x, YEARS, metric, "per yr", B=N_BOOT, seed=SEED)
        pre, post = np.nanmean(x[YEARS <= CHANGE_YEAR]), np.nanmean(x[YEARS > CHANGE_YEAR])
        row.update({f"{prod} | Sen per decade": 10 * tr["sen_slope_per_year"],
                    f"{prod} | 95% CI": f"{10 * tr['sen_ci95_low']:.3g} to {10 * tr['sen_ci95_high']:.3g}",
                    f"{prod} | p (Hamed–Rao)": tr["p_hr"],
                    f"{prod} | mean 2010–2025 minus 1981–2009": post - pre})
    if len(reg_metrics) == 2:
        s2, s3 = row["CHIRPS v2 | Sen per decade"], row["CHIRPS v3 | Sen per decade"]
        p2, p3 = row["CHIRPS v2 | p (Hamed–Rao)"], row["CHIRPS v3 | p (Hamed–Rao)"]
        row["Robust across versions"] = ("yes" if (p2 < .05 and p3 < .05 and np.sign(s2) == np.sign(s3)) else
                                         "no trend in either" if (p2 >= .05 and p3 >= .05) else "no")
    robust_rows.append(row)
robustness = pd.DataFrame(robust_rows)
display(robustness.round(3))
save_table(robustness.round(4), "Table_3_trend_robustness_CHIRPS_v2_v3", "Main",
           "Regional trends (Sen slope per decade with 95% block-bootstrap CI and Hamed–Rao p) and the difference between "
           f"2010–2025 and 1981–{CHANGE_YEAR} means for key metrics in CHIRPS v2 and v3. 'Robust' requires p < 0.05 with the "
           "same sign in both versions.")


# ---- Monthly regional series of independent products (cached, or Earth Engine) ------------
def ee_monthly_regional(name, year, geom):
    ee = ee_init()
    p = PRODUCTS[name]
    imgs = []
    for m in range(1, 13):
        start = ee.Date.fromYMD(year, m, 1)
        col = ee.ImageCollection(p["id"]).filterDate(start, start.advance(1, "month")).select(p["band"])
        img = (col.sum() if p["kind"] == "daily_mm" else ee.Image(col.first()).multiply(1000) if p["kind"] == "monthly_m"
               else ee.Image(col.first()).multiply(24 * calendar.monthrange(year, m)[1]))
        imgs.append(img.rename(f"m{m:02d}"))
    val = ee.Image.cat(imgs).reduceRegion(ee.Reducer.mean(), geom, scale=p["scale"], maxPixels=1e9, tileScale=4).getInfo()
    return [val.get(f"m{m:02d}") for m in range(1, 13)]


def product_monthly(name):
    for folder in [CACHE, *CACHE_CANDIDATES]:
        f = folder / f"interproduct_{name}_monthly.csv"
        if f.is_file():
            return pd.read_csv(f)
    if not ALLOW_EARTH_ENGINE:
        print(f"[skipped] {name}: no cached series")
        return None
    ee = ee_init()
    geom = ee.FeatureCollection(EE_BOUNDARY_ASSET).geometry()
    y0, y1 = PRODUCTS[name]["years"]
    rows = [dict(year=y, month=m + 1, rainfall_mm=v) for y in range(y0, min(y1, END_YEAR) + 1)
            for m, v in enumerate(ee_retry(lambda: ee_monthly_regional(name, y, geom), f"{name} {y}"))]
    df = pd.DataFrame(rows)
    df.to_csv(CACHE / f"interproduct_{name}_monthly.csv", index=False)
    return df


prod_monthly = {"CHIRPS v2": monthly[["year", "month", "rainfall_mm"]].copy()}
if cube_v3 is not None:
    prod_monthly["CHIRPS v3"] = pd.DataFrame(dict(year=np.repeat(YEARS, 12), month=np.tile(np.arange(1, 13), len(YEARS)),
                                                  rainfall_mm=reg_v3.reshape(len(YEARS), 12, 6).sum(axis=2).ravel()))
for name in PRODUCTS:
    df = product_monthly(name)
    if df is not None:
        df = df[["year", "month", "rainfall_mm"]].apply(pd.to_numeric, errors="coerce")
        df.loc[~(df.rainfall_mm >= 0), "rainfall_mm"] = np.nan          # negative / non-finite -> missing
        prod_monthly[name] = df.drop_duplicates(["year", "month"]).sort_values(["year", "month"])


def kge(sim, obs):
    r = np.corrcoef(sim, obs)[0, 1]
    return 1 - np.sqrt((r - 1) ** 2 + (sim.std() / obs.std() - 1) ** 2 + (sim.mean() / obs.mean() - 1) ** 2)


ref_m = prod_monthly["CHIRPS v2"].set_index(["year", "month"]).rainfall_mm
comp_rows, annual_by_product, spi12_by_product = [], {}, {}
for name, df in prod_monthly.items():
    s = df.set_index(["year", "month"]).rainfall_mm
    common = s.index.intersection(ref_m.index)
    pair = pd.DataFrame(dict(p=s.loc[common], c=ref_m.loc[common])).reset_index()
    ok = pair.p.notna()
    ann = pair.groupby("year").agg(p=("p", lambda v: v.sum() if v.notna().all() and len(v) == 12 else np.nan),
                                   c=("c", "sum")).dropna()
    annual_by_product[name] = ann.p
    yrs = ann.index.to_numpy()
    tr = mk_hamed_rao(ann.p.to_numpy(), t=(yrs - yrs[0]).astype(float))
    full = pd.DataFrame(dict(year=np.repeat(np.arange(pair.year.min(), pair.year.max() + 1), 12),
                             month=np.tile(np.arange(1, 13), pair.year.max() - pair.year.min() + 1)))
    full = full.merge(pair, on=["year", "month"], how="left")
    sp, _ = spi(full.p.to_numpy(), full.month.to_numpy(), full.year.to_numpy(), 12)
    sc, _ = spi(full.c.to_numpy(), full.month.to_numpy(), full.year.to_numpy(), 12)
    spi12_by_product[name] = pd.Series(sp, index=pd.to_datetime(dict(year=full.year, month=full.month, day=1)))
    okk = np.isfinite(sp) & np.isfinite(sc)
    pre, post = ann.p[yrs <= CHANGE_YEAR], ann.p[yrs > CHANGE_YEAR]
    comp_rows.append(dict(
        product=name, period=f"{yrs[0]}–{yrs[-1]}", complete_years=len(yrs), missing_months=int((~ok).sum()),
        monthly_r=np.corrcoef(pair.p[ok], pair.c[ok])[0, 1], annual_r=np.corrcoef(ann.p, ann.c)[0, 1],
        monthly_KGE=kge(pair.p[ok].to_numpy(), pair.c[ok].to_numpy()), annual_mean_mm=ann.p.mean(),
        bias_vs_CHIRPSv2_pct=100 * (ann.p.mean() / ann.c.mean() - 1),
        sen_mm_per_decade=10 * tr["sen_slope"], sen_pct_per_decade=100 * 10 * tr["sen_slope"] / ann.p.mean(), trend_p_hr=tr["p_hr"],
        change_after_2009_pct=100 * (post.mean() / pre.mean() - 1) if min(len(pre), len(post)) >= 4 else np.nan,
        pettitt_year=int(yrs[pettitt(ann.p.to_numpy())["tau_index"]]), pettitt_p=pettitt(ann.p.to_numpy())["p_approx"],
        driest_year=int(ann.p.idxmin()), spi12_r=np.corrcoef(sp[okk], sc[okk])[0, 1],
        spi12_minimum_month=spi12_by_product[name].idxmin().strftime("%Y-%m")))
products = pd.DataFrame(comp_rows)
display(products.round(3))
save_table(products.round(4), "Table_4_precipitation_product_agreement", "Main",
           "Agreement of CHIRPS v2 with CHIRPS v3, PERSIANN-CDR, ERA5-Land and IMERG V07 over each product's overlap "
           "(area-weighted regional means): correlation, Kling–Gupta efficiency and bias relative to CHIRPS v2, Sen trend "
           "(Hamed–Rao p), change of the mean after 2009, Pettitt year, driest year and SPI-12 agreement. Agreement does not "
           "imply accuracy; no gauge data were available.")

# ---- Figure: product agreement ------------------------------------------------------------
fig = new_figure("double", 92)
gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1])
ax = fig.add_subplot(gs[0])
base = (2001, 2020)
for name, a in annual_by_product.items():
    idx = 100 * a / a.loc[base[0]:base[1]].mean()
    ax.plot(idx.index, idx.values, color=PRODUCT_COLORS[name], lw=1.3 if name == "CHIRPS v2" else 0.9, label=name)
ax.axhline(100, color=PAL["muted"], lw=0.5, ls="--")
ax.axvline(CHANGE_YEAR + 0.5, color=PAL["grey"], lw=0.6, ls=":")
ax.set(ylabel=f"Annual rainfall (% of {base[0]}–{base[1]} mean)", xlim=(YEARS[0] - 1, YEARS[-1] + 1))
ax.legend(ncol=3, loc="upper left", fontsize=8)
panel_label(ax, "a")
ax = fig.add_subplot(gs[1])
pp = products.set_index("product")
y = np.arange(len(pp))
ax.barh(y, pp.change_after_2009_pct, color=[PRODUCT_COLORS[n] for n in pp.index], height=0.6)
for yi, (n, r) in enumerate(pp.iterrows()):
    ax.text(r.change_after_2009_pct + 1.5, yi, f"{r.change_after_2009_pct:.0f}%", va="center", fontsize=8)
ax.set_yticks(y, pp.index)
ax.invert_yaxis()
ax.set_xlabel(f"Mean {CHANGE_YEAR + 1}–2025 vs {START_YEAR}–{CHANGE_YEAR} (%)")
ax.set_xlim(0, pp.change_after_2009_pct.max() * 1.25)
panel_label(ax, "b")
save_figure(fig, "Fig08_precipitation_product_agreement", "Main",
            f"Consistency of the rainfall signal across precipitation products (area-weighted Cholistan means). (a) Annual "
            f"totals as % of each product's {base[0]}–{base[1]} mean; dotted line: {CHANGE_YEAR}/{CHANGE_YEAR + 1}. (b) Change "
            f"of the mean after {CHANGE_YEAR} relative to the earlier period (IMERG from 2001, PERSIANN-CDR from 1983). "
            f"Driest year in all products: {', '.join(sorted(set(str(v) for v in products.driest_year)))}.")

# ---- Supplementary figure: dry-pentad definition sensitivity incl. CHIRPS v3 -------------------------
fig = new_figure("double", 115)
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.05], height_ratios=[1, 1])
ax = fig.add_subplot(gs[0, 0])
ax.bar(YEARS, attrib.rain_bearing, color=PAL["purple_dark"], width=0.8, label=f"Rain-bearing pentads (normal ≥{NEAR_RAINLESS_MM:g} mm)")
ax.bar(YEARS, attrib.near_rainless, bottom=attrib.rain_bearing, color=PAL["purple_light"], width=0.8,
       label=f"Near-rainless pentads (normal <{NEAR_RAINLESS_MM:g} mm)")
ax.set(ylabel="P20 dry pentads (CHIRPS v2)", xlim=(YEARS[0] - 1, YEARS[-1] + 1))
ax.legend(loc="upper left", fontsize=8)
panel_label(ax, "a")
ax = fig.add_subplot(gs[1, 0])
ax.plot(YEARS, reg_metrics["CHIRPS v2"]["P20 dry pentads, near-rainless pentads (count)"], color=PRODUCT_COLORS["CHIRPS v2"],
        marker="o", ms=2, label="CHIRPS v2")
if "CHIRPS v3" in reg_metrics:
    ax.plot(YEARS, reg_metrics["CHIRPS v3"]["P20 dry pentads, near-rainless pentads (count)"], color=PRODUCT_COLORS["CHIRPS v3"],
            marker="s", ms=2, label="CHIRPS v3")
ax.set(ylabel="Near-rainless P20 dry pentads", xlabel="Year", xlim=(YEARS[0] - 1, YEARS[-1] + 1))
ax.legend(loc="upper left")
panel_label(ax, "b")
ax = fig.add_subplot(gs[:, 1])
show = dry_sens[dry_sens.metric == "longest run"]
order = list(DRY_DEFINITIONS)
for s, (col, mk, off) in {"Annual": (PAL["purple"], "o", -0.13), "JJAS": (PAL["ink"], "D", 0.13)}.items():
    g = show[show.season == s].set_index("definition").loc[order]
    yy = np.arange(len(order)) + off
    sig = (g.q_bh_within_definition < 0.05).to_numpy()
    ax.errorbar(10 * g.sen_slope_per_year, yy, xerr=[10 * (g.sen_slope_per_year - g.sen_ci95_low),
                10 * (g.sen_ci95_high - g.sen_slope_per_year)], fmt="none", ecolor=col, elinewidth=0.8)
    ax.scatter(10 * g.sen_slope_per_year[~sig], yy[~sig], marker=mk, s=12, facecolor="white", edgecolor=col, lw=0.8, zorder=3)
    ax.scatter(10 * g.sen_slope_per_year[sig], yy[sig], marker=mk, s=12, color=col, zorder=3, label=s)
ax.axvline(0, color=PAL["muted"], lw=0.5)
ax.set_yticks(np.arange(len(order)), order)
ax.invert_yaxis()
ax.set_xlabel("Trend in longest dry run (pentads decade$^{-1}$)")
ax.legend(loc="upper left")
panel_label(ax, "c")
_v3txt = ""
if "CHIRPS v3" in reg_metrics:
    _v2n, _v3n = (reg_metrics[p]["P20 dry pentads, near-rainless pentads (count)"] for p in ("CHIRPS v2", "CHIRPS v3"))
    _v3txt = (f" Mean near-rainless dry pentads 1981–1999 → 2000–2025: CHIRPS v2 {_v2n[YEARS < 2000].mean():.1f} → "
              f"{_v2n[YEARS >= 2000].mean():.1f}; CHIRPS v3 {_v3n[YEARS < 2000].mean():.1f} → {_v3n[YEARS >= 2000].mean():.1f}.")
save_figure(fig, "FigS1_dry_pentad_definition_sensitivity", "Supplementary",
            f"Dependence of dry-pentad trends on the definition. (a) Annual P20 dry pentads (CHIRPS v2) split by "
            f"climatological regime; {attrib.near_rainless.sum() / attrib.all_pentads.sum():.0%} fall in calendar pentads "
            f"whose 1991–2020 mean is <{NEAR_RAINLESS_MM:g} mm. (b) Near-rainless P20 dry pentads in CHIRPS v2 and v3; "
            f"homogeneity tests place a break in v2 at {int(hom.change_year.iloc[0])}. (c) Sen trends (95% block-bootstrap "
            f"CI) of the longest dry run under eight definitions; filled: BH q < 0.05." + _v3txt)
print("Saved Fig08 and FigS1.")

## 14. Exploratory rainfall zones
Data-driven zones replace a hand-drawn Greater/Lesser Cholistan division, for which no authoritative boundary exists. They are reported as exploratory unless the year-resampling stability (adjusted Rand index) is ≥ 0.6.

In [ ]:
# =============================================================================
# 14. Exploratory objective rainfall zones (supplementary)
# -----------------------------------------------------------------------------
# No authoritative Greater/Lesser Cholistan boundary exists. Cells are grouped by the
# similarity of their standardized annual-rainfall anomalies (co-variability, not
# trend) with spatially constrained Ward clustering; k is chosen by the silhouette
# criterion (≥5 % area per zone) and stability is assessed by resampling years
# (adjusted Rand index). Zones are reported as exploratory unless stable.
# =============================================================================
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

K_RANGE = range(2, 9)
MIN_ZONE_AREA_PCT = 5.0
N_STABILITY = 200


def zone_features(year_idx=None):
    X = metrics_v2["Annual rainfall (mm)"][1]                  # (Y, P)
    if year_idx is not None:
        X = X[year_idx]
    Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
    return Z.T                                                 # (P, Y)


def constrained_ward(F, k):
    return AgglomerativeClustering(n_clusters=k, linkage="ward", connectivity=A_queen).fit_predict(F)


F = zone_features()
k_rows, labels_by_k = [], {}
for k in K_RANGE:
    lab = constrained_ward(F, k)
    labels_by_k[k] = lab
    shares = np.array([pix_w[lab == z].sum() for z in range(k)]) / pix_w.sum() * 100
    k_rows.append(dict(k=k, silhouette=silhouette_score(F, lab), calinski_harabasz=calinski_harabasz_score(F, lab),
                       davies_bouldin=davies_bouldin_score(F, lab), smallest_zone_area_pct=shares.min()))
k_select = pd.DataFrame(k_rows)
eligible = k_select[k_select.smallest_zone_area_pct >= MIN_ZONE_AREA_PCT]
K_BEST = int(eligible.loc[eligible.silhouette.idxmax(), "k"])

rng = np.random.default_rng(SEED)
ari = {k: [] for k in K_RANGE}
for _ in range(N_STABILITY):
    yi = np.sort(rng.choice(len(YEARS), len(YEARS), replace=True))
    Fb = zone_features(yi)
    for k in K_RANGE:
        ari[k].append(adjusted_rand_score(labels_by_k[k], constrained_ward(Fb, k)))
k_select["stability_mean_ARI"] = [np.mean(ari[k]) for k in K_RANGE]
k_select["stability_ARI_p05"] = [np.quantile(ari[k], 0.05) for k in K_RANGE]
k_select["selected"] = k_select.k == K_BEST
display(k_select.round(3))

# Order zones north -> south so labels are reproducible
raw = labels_by_k[K_BEST]
order = np.argsort([-pix_lat[raw == z].mean() for z in range(K_BEST)])
zone = np.empty_like(raw)
for new, old in enumerate(order):
    zone[raw == old] = new
ZONE_NAMES = [f"Z{i + 1}" for i in range(K_BEST)]
print(f"Selected k = {K_BEST} contiguous zones (silhouette criterion, ≥{MIN_ZONE_AREA_PCT:g}% area each).")
_ari = k_select.loc[k_select.selected, "stability_mean_ARI"].iloc[0]
if _ari < 0.6:
    print(f"WARNING: mean year-resampling ARI = {_ari:.2f} (<0.6). Zone boundaries are not robust; "
          "present the zones as exploratory and rely on the pixel maps/LISA for spatial inference.")

# Cross-check: same k without the contiguity constraint (are zones forced by it?)
uncon = AgglomerativeClustering(n_clusters=K_BEST, linkage="ward").fit_predict(F)
print(f"Agreement with unconstrained Ward (ARI): {adjusted_rand_score(zone, uncon):.2f}")


# ---- Zone-level tests -------------------------------------------------------------
R_all = cube_v2[:, :, pix_r, pix_c].astype(float)                       # (Y, 72, P)
zone_rows, zone_series = [], {}
for zi, zn in enumerate(ZONE_NAMES):
    m = zone == zi
    w = pix_w[m] / pix_w[m].sum()
    Rz = (R_all[:, :, m] * w).sum(axis=-1)[:, :, None]                  # (Y, 72, 1)
    monz = Rz[:, :, 0].reshape(len(YEARS), 12, 6).sum(axis=2)
    series = {"Annual rainfall (mm)": Rz[:, :, 0].sum(1),
              "JJAS rainfall (mm)": monz[:, 5:9].sum(1),
              "Wet pentads ≥5 mm (count)": concentration_metrics(Rz, WET_PENTAD_MM)["n_wet_pentads_ge5"][:, 0],
              "JJAS longest dry run <2 mm (pentads)": pentad_dryness(Rz, YEARS, DRY_DEFINITIONS["Absolute <2 mm"],
                                                                     NORMAL, months=SEASONS["JJAS"])[1][:, 0],
              "P20 dry pentads, normal ≥1 mm (count)": pentad_dryness(Rz, YEARS, DRY_DEFINITIONS["P20 (normal ≥1 mm)"],
                                                                       NORMAL)[0][:, 0]}
    s12, _ = spi(monz.ravel(), np.tile(np.arange(1, 13), len(YEARS)), np.repeat(YEARS, 12), 12)
    series["SPI-12 annual minimum (index)"] = np.r_[np.nan, np.nanmin(s12.reshape(len(YEARS), 12)[1:], axis=1)]
    zone_series[zn] = series
    cp = pettitt(series["Annual rainfall (mm)"])
    for sname, x in series.items():
        r = trend_row(x, YEARS, sname, "per yr", B=N_BOOT, seed=SEED)
        r.update(zone=zn, area_km2=pix_w[m].sum() / 1e6, cells=int(m.sum()),
                 centroid_lon=np.average(pix_lon[m], weights=pix_w[m]), centroid_lat=np.average(pix_lat[m], weights=pix_w[m]),
                 normal_1991_2020=np.nanmean(x[ref]) if sname.endswith("(mm)") else np.nan,
                 pettitt_year_annual=int(YEARS[cp["tau_index"]]), pettitt_p_annual=cp["p_approx"])
        zone_rows.append(r)
zone_trends = pd.DataFrame(zone_rows)
zone_trends["q_bh_within_metric"] = zone_trends.groupby("series")["p_hr"].transform(bh_adjust)
display(zone_trends[["zone", "series", "area_km2", "centroid_lat", "normal_1991_2020", "sen_slope_per_year",
                     "sen_ci95_low", "sen_ci95_high", "p_hr", "q_bh_within_metric", "pettitt_year_annual"]].round(3))


def slope_difference_ci(x1, x2, B=N_BOOT, seed=SEED):
    """Joint moving-block bootstrap (same year blocks for both zones) of the
    difference in Sen slopes, preserving inter-zone correlation."""
    n = len(x1); t = np.arange(n, dtype=float)
    (b1, a1), (b2, a2) = sen_slope(x1, t), sen_slope(x2, t)
    r1, r2 = x1 - a1 - b1 * t, x2 - a2 - b2 * t
    idx = _mbb_indices(n, B, max(2, int(round(n ** (1 / 3)))), np.random.default_rng(seed))
    d = sen_slope((a1 + b1 * t)[:, None] + r1[idx].T, t)[0] - sen_slope((a2 + b2 * t)[:, None] + r2[idx].T, t)[0]
    return b1 - b2, *np.quantile(d, [0.025, 0.975])


pair_rows = []
for i in range(K_BEST):
    for j in range(i + 1, K_BEST):
        for sname in ("Annual rainfall (mm)", "JJAS rainfall (mm)", "JJAS longest dry run <2 mm (pentads)"):
            d, lo, hi = slope_difference_ci(zone_series[ZONE_NAMES[i]][sname], zone_series[ZONE_NAMES[j]][sname])
            pair_rows.append(dict(contrast=f"{ZONE_NAMES[i]} − {ZONE_NAMES[j]}", series=sname,
                                  slope_difference_per_year=d, ci95_low=lo, ci95_high=hi, excludes_zero=(lo > 0) or (hi < 0)))
zone_contrasts = pd.DataFrame(pair_rows)
display(zone_contrasts.round(3))

save_table(k_select.round(4), "Table_S29_zone_number_selection", "Supplementary",
           "Spatially constrained Ward clustering of standardized annual-rainfall anomalies: silhouette, Calinski–Harabasz and "
           f"Davies–Bouldin indices, smallest zone area and year-resampling stability (adjusted Rand index, {N_STABILITY} replicates).")
save_table(zone_trends.round(4), "Table_S30_zone_trends", "Supplementary",
           "Regional tests repeated for each exploratory zone (area-weighted zone means).")
save_table(zone_contrasts.round(4), "Table_S31_zone_slope_contrasts", "Supplementary",
           "Differences in Sen slopes between zones with joint moving-block bootstrap 95% CIs.")

ZONE_COLORS = ["#91278F", "#2B8C8C", "#C9822B", "#4F6DB8", "#7A9A3A", "#B5475B", "#6A626A", "#D4A017"]
fig = new_figure("double", 78)
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1])
ax = fig.add_subplot(gs[0])
ax.pcolormesh(lon_edges, lat_edges, to_grid(zone.astype(float)), cmap=ListedColormap(ZONE_COLORS[:K_BEST]),
              norm=BoundaryNorm(np.arange(-0.5, K_BEST + 0.5), K_BEST), shading="flat", rasterized=True)
style_map(ax)
for zi, zn in enumerate(ZONE_NAMES):
    m = zone == zi
    ax.text(np.average(pix_lon[m], weights=pix_w[m]), np.average(pix_lat[m], weights=pix_w[m]), zn, ha="center",
            va="center", fontsize=8, fontweight="bold", color="white")
panel_label(ax, "a", x=-0.03)
ax = fig.add_subplot(gs[1])
for zi, zn in enumerate(ZONE_NAMES):
    ax.plot(YEARS, zone_series[zn]["Annual rainfall (mm)"], color=ZONE_COLORS[zi], label=zn)
ax.set(ylabel="Annual rainfall (mm)", xlim=(YEARS[0] - 1, YEARS[-1] + 1))
ax.legend(ncol=min(K_BEST, 4), loc="upper left")
panel_label(ax, "b")
_sel = k_select[k_select.selected].iloc[0]
save_figure(fig, "FigS4_exploratory_rainfall_zones", "Supplementary",
            f"Exploratory rainfall zones. (a) {K_BEST} contiguous zones from spatially constrained Ward clustering of "
            f"standardized annual-rainfall anomalies (silhouette {_sel.silhouette:.2f}; year-resampling ARI "
            f"{_sel.stability_mean_ARI:.2f}). (b) Area-weighted annual rainfall per zone.")
print("Saved zone outputs.")

## 15–16. Study-area map and participant-derived priorities

In [ ]:
# =============================================================================
# 15. Study-area map (OpenStreetMap context when available)
# -----------------------------------------------------------------------------
# Optional inputs: geopandas, and the Geofabrik Pakistan extract
# (https://download.geofabrik.de/asia/pakistan.html, 'pakistan-latest-free.shp.zip')
# at OSM_ZIP. Optional 03_GIS/consultation_points.csv (lon, lat) for consultation
# localities — generalise to village/union-council level before publication.
# Without them, a boundary-and-grid map is produced under the same name.
# =============================================================================
CONSULTATION_CSV = PROJECT_ROOT / "03_GIS" / "consultation_points.csv"
try:
    import geopandas as gpd
    HAVE_OSM = OSM_ZIP.is_file()
except ImportError:
    HAVE_OSM = False

(minx, miny), (maxx, maxy) = _xy.min(axis=0), _xy.max(axis=0)
bbox = (minx - 0.3, miny - 0.3, maxx + 0.3, maxy + 0.3)
fig = new_figure("double", 125)
ax = fig.add_subplot(111)
ax.vlines(lon_edges, lat_edges.min(), lat_edges.max(), color="#D5D0D5", lw=0.15, zorder=1)
ax.hlines(lat_edges, lon_edges.min(), lon_edges.max(), color="#D5D0D5", lw=0.15, zorder=1)
ax.pcolormesh(lon_edges, lat_edges, np.where(analysis_mask, 1.0, np.nan), cmap=ListedColormap([PAL["purple_light"]]),
              shading="flat", alpha=0.6, zorder=1, rasterized=True)
handles = [Line2D([], [], color=PAL["ink"], lw=1, label="Cholistan boundary"),
           Patch(facecolor=PAL["purple_light"], alpha=0.6, label=f"CHIRPS 0.05° cells analysed (≥{ANALYSIS_MIN_FRACTION:.0%} inside)")]
if HAVE_OSM:
    def osm(layer):
        return gpd.read_file(f"zip://{OSM_ZIP}!{layer}.shp", bbox=bbox).to_crs("EPSG:4326")
    bnd = gpd.read_file(BOUNDARY_SHP).to_crs("EPSG:4326")
    ww = osm("gis_osm_waterways_free_1")
    places = osm("gis_osm_places_free_1")
    water = gpd.clip(osm("gis_osm_water_a_free_1"), bnd)
    water = water[water.to_crs("EPSG:32643").area < 20e4]
    roads = osm("gis_osm_roads_free_1")
    roads[roads.fclass.isin(["motorway", "trunk", "primary", "secondary"])].plot(ax=ax, color="#8C8C8C", lw=0.4, zorder=2)
    ww[ww.fclass.isin(["canal", "drain"])].plot(ax=ax, color="#3A7DBF", lw=0.45, zorder=3)
    ww[ww.fclass == "river"].plot(ax=ax, color="#1F4E8C", lw=1.0, zorder=3)
    if len(water):
        c = water.to_crs("EPSG:32643").centroid.to_crs("EPSG:4326")
        ax.scatter(c.x, c.y, s=3, marker="s", color="#2B8C8C", lw=0, zorder=4)
    vil = places[places.fclass.isin(["village", "hamlet"])]
    ax.scatter(vil.geometry.x, vil.geometry.y, s=0.8, color=PAL["muted"], lw=0, zorder=4)
    towns = places[places.fclass.isin(["city", "town"])]
    ax.scatter(towns.geometry.x, towns.geometry.y, s=9, color=PAL["ink"], zorder=6)
    for _, r in towns.iterrows():
        if isinstance(r.get("name"), str):
            ax.text(r.geometry.x + 0.025, r.geometry.y + 0.015, r["name"], fontsize=8, zorder=7)
    handles += [Line2D([], [], color="#1F4E8C", lw=1.0, label="River"), Line2D([], [], color="#3A7DBF", lw=0.6, label="Canal / drain"),
                Line2D([], [], color="#8C8C8C", lw=0.6, label="Main road"),
                Line2D([], [], marker="o", ls="", color=PAL["ink"], ms=3, label="Town"),
                Line2D([], [], marker="o", ls="", color=PAL["muted"], ms=1.5, label="Village / hamlet"),
                Line2D([], [], marker="s", ls="", color="#2B8C8C", ms=2.5, label="Small water body <20 ha (Toba candidate)")]
    source = "Basemap: © OpenStreetMap contributors (ODbL), Geofabrik extract."
else:
    source = "OSM layers not available; boundary and CHIRPS grid only."
    print("geopandas or the Geofabrik extract not found: drawing the boundary-and-grid version.")
if CONSULTATION_CSV.is_file():
    cp = pd.read_csv(CONSULTATION_CSV)
    ax.scatter(cp.lon, cp.lat, s=28, marker="*", color="#B2182B", edgecolor="white", lw=0.3, zorder=8)
    handles.append(Line2D([], [], marker="*", ls="", color="#B2182B", ms=6, label="Consultation locality"))
for ring in rings:
    if not ring_is_hole(ring):
        ax.plot(ring[:, 0], ring[:, 1], color=PAL["ink"], lw=1.0, zorder=5)
ax.set_aspect(1 / np.cos(np.deg2rad(lat_c.mean())))
ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.1f}°E"); ax.yaxis.set_major_formatter(lambda y, _: f"{y:.1f}°N")
for s in ("top", "right"):
    ax.spines[s].set_visible(True)
km = 50 / (111.32 * np.cos(np.deg2rad(bbox[1] + 0.2)))
ax.plot([bbox[0] + 0.12, bbox[0] + 0.12 + km], [bbox[1] + 0.12] * 2, color=PAL["ink"], lw=1.6, zorder=9)
ax.text(bbox[0] + 0.12 + km / 2, bbox[1] + 0.16, "50 km", ha="center", fontsize=8)
ax.annotate("N", xy=(0.96, 0.95), xytext=(0.96, 0.85), xycoords="axes fraction", ha="center", fontsize=8, fontweight="bold",
            arrowprops=dict(arrowstyle="-|>", lw=0.8, color=PAL["ink"]))
ax.legend(handles=handles, loc="lower right", fontsize=8, frameon=True, facecolor="white", edgecolor=PAL["grey_light"], framealpha=0.95)
save_figure(fig, "Fig01_study_area", "Main",
            f"Cholistan Desert study area (boundary area {weights.sum() / 1e6:,.0f} km²) with the native CHIRPS 0.05° grid; "
            f"shaded cells (n = {int(analysis_mask.sum())}) are ≥{ANALYSIS_MIN_FRACTION:.0%} inside the boundary and form the "
            f"spatial analysis domain. {source}")
print("Saved Fig01.")

# =============================================================================
# 16. Participant-derived resilience priorities (descriptive; not measured outcomes)
# =============================================================================
table2 = pd.DataFrame([
    ["Greater Cholistan", "Remoteness; saline/brackish groundwater; weak road and water infrastructure; restricted service and forage access",
     "Priority drought support; strategic road/water access; mobile veterinary services; pre-positioned emergency logistics"],
    ["Lesser Cholistan", "Contraction and fragmentation of common grazing resources with agricultural expansion",
     "Protect grazing access and movement options; locally negotiated rangeland management"],
    ["Tobas and water", "Siltation, poor maintenance and unreliable water supply",
     "Rehabilitation/desilting; community management; reliable solar-powered pumping where technically appropriate"],
    ["Rangelands and forage", "Degraded grazing resources and insufficient drought reserves",
     "Pilot reseeding/rehabilitation with locally adapted drought-tolerant forage, grasses, shrubs and trees; strategic fodder reserves"],
    ["Livestock genetic resources", "Concern about deterioration of locally adapted livestock characteristics",
     "Conserve Cholistani cattle and evaluate pedigree-bull, recording and appropriate breeding support"],
    ["Emergency feed", "Feed scarcity and high transport cost during drought",
     "Evaluate locally produced crop-residue feed blocks and decentralised strategic feed reserves"],
    ["Emergency finance", "Distress sale of productive livestock to meet urgent household and animal needs",
     "Assess feasible concessional/interest-free emergency finance and other early-liquidity mechanisms"],
    ["Veterinary support", "Difficulty reaching remote livestock populations during crises",
     "Mobile veterinary services, vaccination/treatment preparedness and drought advisories"],
    ["Community and institutions", "Fragmented preparedness and weak coordination",
     "Stronger community responsibility and coordination among livestock, drought-management, government, NGO and humanitarian actors"],
], columns=["Domain", "Participant-reported vulnerability or constraint", "Participant-proposed resilience priority"])
save_table(table2, "Table_2_participant_derived_resilience_priorities", "Main",
           "Resilience priorities reported by participants in the consultations (descriptive synthesis; not measured "
           "livestock, forage or livelihood outcomes). 'Greater' and 'Lesser' Cholistan are participants' own terms.")
display(table2)

## 17. Exports, figure quality control and provenance
Every TIFF is re-opened and checked for 600 dpi, RGB colour, LZW compression, a width equal to the declared journal width and a height ≤ 230 mm. The notebook stops if any figure fails.

In [ ]:
# =============================================================================
# 17. GIS rasters, captions, manifest, figure QC, key results and provenance
# =============================================================================
from importlib import metadata as _md
from PIL import Image

# ---- Native-resolution GeoTIFFs (optional: rasterio) ---------------------------------
try:
    import rasterio
    from rasterio.transform import from_origin
    _tf = from_origin(lon_edges[0], lat_edges[0], grid_meta["sx"], -grid_meta["sy"])
    _layers = {}
    for n, m in tmaps_v2.items():
        key = n.split(" (")[0].replace(" ", "_").replace("≥", "ge").replace("<", "lt").replace(",", "").replace("-", "")
        _layers[f"{key}_slope_per_decade"] = m["slope_per_decade"]
        _layers[f"{key}_q_BH"] = m["q"]
    for n, L in lisa_maps.items():
        key = n.split(" (")[0].replace(" ", "_").replace("≥", "ge").replace("-", "")
        codes = {"High–High": 1, "Low–Low": 2, "High–Low": 3, "Low–High": 4}
        _layers[f"{key}_LISA_class"] = np.array([codes.get(c, 0) if q <= 0.05 else 0 for c, q in zip(L["quadrant"], L["q_bh"])], float)
    for lname, v in _layers.items():
        with rasterio.open(DIRS["gis"] / f"{lname}.tif", "w", driver="GTiff", height=Hg, width=Wg, count=1, dtype="float32",
                           crs=grid_meta["crs"], transform=_tf, nodata=-9999, compress="deflate") as dst:
            dst.write(np.nan_to_num(to_grid(v), nan=-9999).astype("float32"), 1)
    print(f"{len(_layers)} GeoTIFFs written (LISA class: 1 HH, 2 LL, 3 HL, 4 LH, 0 n.s.).")
except ImportError:
    print("rasterio not installed: GeoTIFFs skipped (Table S28 holds every cell value with coordinates).")

# ---- Captions and manifest ------------------------------------------------------------------
def _order(name):
    return (0 if name.startswith(("Fig0", "Table_1", "Table_2", "Table_3", "Table_4")) else 1, name)

lines = ["# Figure and table captions (generated from the analysis)\n"]
for title, reg in (("Figures", FIGURES), ("Tables", TABLES)):
    for role in ("Main", "Supplementary"):
        lines.append(f"\n## {role} {title.lower()}\n")
        for it in sorted([r for r in reg if r["role"] == role], key=lambda r: _order(r["name"])):
            lines.append(f"**{it['name']}.** {it['caption']}\n")
(OUT / "Captions.md").write_text("\n".join(lines), encoding="utf-8")
manifest = pd.concat([pd.DataFrame(FIGURES).assign(kind="figure"), pd.DataFrame(TABLES).assign(kind="table")], ignore_index=True)
manifest.sort_values(["role", "kind", "name"]).to_csv(OUT / "Output_manifest.csv", index=False)

# ---- Figure quality control: inspect the written files -----------------------------------------
qc = []
for it in FIGURES:
    with Image.open(OUT / it["file"]) as im:
        w_mm, h_mm = im.width / FIG_DPI * 25.4, im.height / FIG_DPI * 25.4
        dpi = im.info.get("dpi", (0, 0))
        checks = dict(dpi_600=min(dpi) >= 599.5, rgb=im.mode == "RGB", lzw=im.info.get("compression") == "tiff_lzw",
                      width_ok=abs(w_mm - FIG_WIDTH_MM[it["width_class"]]) < 0.5, height_ok=h_mm <= FIG_MAX_HEIGHT_MM + 0.5)
    qc.append(dict(figure=it["name"], role=it["role"], width_mm=round(w_mm, 1), height_mm=round(h_mm, 1), pixels=f"{im.width}×{im.height}",
                   dpi=round(min(dpi)), mode=im.mode, **checks, status="PASS" if all(checks.values()) else "FAIL"))
qc = pd.DataFrame(qc)
qc.to_csv(OUT / "Figure_quality_check.csv", index=False)
display(qc)
assert (qc.status == "PASS").all(), "A figure failed the publication checks"

# ---- Key results, assembled from the computed tables (for the Results section) --------------------
rt = trends_all.set_index("series")
key = [
    f"Annual rainfall: Sen slope {rt.loc['Annual rainfall', 'sen_slope_per_year']:.2f} mm/yr "
    f"(95% CI {rt.loc['Annual rainfall', 'sen_ci95_low']:.2f}–{rt.loc['Annual rainfall', 'sen_ci95_high']:.2f}; "
    f"Hamed–Rao p = {rt.loc['Annual rainfall', 'p_hr']:.1e}).",
    f"No trend within 1981–2008 (p = {subperiods.query('period == \"1981–2008\" and series == \"Annual rainfall\"').p_hr.iloc[0]:.2f}) "
    f"or 2010–2025 (p = {subperiods.query('period == \"2010–2025\" and series == \"Annual rainfall\"').p_hr.iloc[0]:.2f}); "
    f"after removing the {CHANGE_YEAR} mean shift, p = {subperiods[subperiods.period.str.contains('removed') & (subperiods.series == 'Annual rainfall')].p_hr.iloc[0]:.2f}.",
    f"Best model: {cp_models.model.iloc[0]} (BIC weight {cp_models.BIC_weight.iloc[0]:.2f}); change-year 95% range "
    f"{cp_unc['change_year_ci95'][0]}–{cp_unc['change_year_ci95'][1]}; Pettitt rejects in "
    f"{cp_unc['pettitt_rejection_rate_under_pure_trend']:.0%} of pure-trend simulations.",
    f"Wet pentads (≥{WET_PENTAD_MM:g} mm): frequency {decomp.pct_per_decade.iloc[1]:+.1f}%/decade, intensity "
    f"{decomp.pct_per_decade.iloc[2]:+.1f}%/decade.",
    f"P20 dry pentads: {attrib.near_rainless.sum() / attrib.all_pentads.sum():.0%} in near-rainless calendar pentads; "
    f"rain-bearing component trend p = {attrib_tr.p_hr.iloc[2]:.2f}; JJAS longest run <2 mm Sen "
    f"{10 * dry_sens.query('definition == \"Absolute <2 mm\" and season == \"JJAS\" and metric == \"longest run\"').sen_slope_per_year.iloc[0]:+.2f} pentads/decade.",
    f"Spatial: annual-rainfall increase significant (q ≤ 0.10) over "
    f"{trend_summary_v2.set_index('metric').loc['Annual rainfall (mm)', 'pct_area_sig_increase_q10']:.0f}% of the area.",
    f"Products: change after {CHANGE_YEAR} " + ", ".join(f"{r.product} {r.change_after_2009_pct:+.0f}%" for r in products.itertuples()
                                                         if np.isfinite(r.change_after_2009_pct))
    + f"; driest year {', '.join(sorted(set(str(v) for v in products.driest_year)))}.",
]
if "Robust across versions" in robustness:
    key.append("Not robust between CHIRPS v2 and v3: " + (", ".join(robustness.loc[robustness["Robust across versions"] == "no", "metric"]) or "none") + ".")
(OUT / "Key_results.md").write_text("# Key results (auto-generated)\n\n" + "\n".join(f"- {k}" for k in key) + "\n", encoding="utf-8")
print("\n".join(key))

# ---- Provenance ---------------------------------------------------------------------------------------
prov = dict(created=pd.Timestamp.now().isoformat(timespec="seconds"), python=sys.version.split()[0], platform=platform.platform(),
            inputs=dict(regional_series=str(PENTAD_CACHE_CSV.relative_to(PROJECT_ROOT)), boundary=str(BOUNDARY_SHP.relative_to(PROJECT_ROOT)),
                        spatial_cache=str(CACHE)),
            datasets=dict(primary=CHIRPS_V2_PENTAD, version_sensitivity=CHIRPS_V3_PENTAD, products={k: v["id"] for k, v in PRODUCTS.items()}),
            settings=dict(period=[START_YEAR, END_YEAR], normal=list(NORMAL), spi_calibration="full record",
                          wet_pentad_mm=WET_PENTAD_MM, analysis_min_fraction=ANALYSIS_MIN_FRACTION,
                          n_boot=N_BOOT, n_perm=N_PERM, lisa_permutations=LISA_PERMUTATIONS, seed=SEED),
            packages={p: (_md.version(p) if p in {d.metadata["Name"] for d in _md.distributions()} else "not installed")
                      for p in ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "pillow", "earthengine-api", "geopandas", "rasterio")},
            outputs=dict(figures=len(FIGURES), tables=len(TABLES)))
(OUT / "Provenance.json").write_text(json.dumps(prov, indent=2))
print(f"\nDone: {len(FIGURES)} figures and {len(TABLES)} tables in {OUT}")

---
## Interpretation notes
1. **Measured outcomes.** Only precipitation was measured. Participant priorities (Table 2) are descriptive and are not measured livestock, forage or livelihood outcomes.
2. **Shift, not steady trend.** The 1981–2025 increase is concentrated after the mid-2000s, and each sub-period alone shows no trend. Report it as a shift whose timing is uncertain (Fig. 4, Tables S13–S15), not as a regime change in a single year.
3. **Dry-pentad persistence.** Increases in percentile dry pentads counted over all calendar pentads come from near-rainless pentads (Fig. S1). Hydrologically meaningful definitions show no increase, or shorter dry spells.
4. **Robustness.** Interpret only signals that hold in both CHIRPS versions (Table 3) and in the product comparison (Table 4).
5. **Spatial clusters** show where trends are relatively stronger or weaker. Exploratory zones are not a substitute for an administrative Greater/Lesser boundary.